In [1]:
# Cell 1 — Imports & Environment Check
import subprocess, sys

# ── Standard library ──────────────────────────────────────────────
import os, json, re, ast, warnings, logging
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter

# ── Scientific stack ──────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── sklearn metrics ───────────────────────────────────────────────
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (f1_score,
                             label_ranking_average_precision_score,
                             label_ranking_loss,
                             coverage_error)

# ── PyTorch ───────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

# ── HuggingFace ───────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModel

# ── Misc ──────────────────────────────────────────────────────────
import openpyxl
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ── Environment ───────────────────────────────────────────────────
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}  "
      f"({'  ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")

import random
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Config ────────────────────────────────────────────────────────
from types import SimpleNamespace
CFG = SimpleNamespace(
    encoder_name      = "cisco-ai/SecureBERT2.0-base",
    max_len           = 512,
    batch_size        = 64,       # inference only, can be large
    num_workers       = 0,
    pin_memory        = False,

    # Paths
    attack_json       = "OSRs/ATTACK/enterprise-attack-v16.1.json",
    kev_json          = "OSRs/kev-07.28.2025_attack-16.1-enterprise.json",
    smet_xlsx         = "CVE_annotated_dataset.xlsx",
    smet_id2mitre_url = "https://raw.githubusercontent.com/basel-a/SMET/main/id2mitre.json",

    # Eval
    top_k_list        = [1, 3, 5, 10],   # R@K values to report
)

# Validate paths
print("\nPaths:")
for label, path in [("ATT&CK STIX", CFG.attack_json),
                    ("KEV gold",    CFG.kev_json),
                    ("SMET xlsx",   CFG.smet_xlsx)]:
    ok = Path(path).exists()
    size = f"{Path(path).stat().st_size/1e6:.1f} MB" if ok else "MISSING"
    print(f"  {'✓' if ok else '✗'}  {label:<14} {path}  ({size})")

print("\n✓ Ready for Cell 2.")

c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python  : 3.11.0
PyTorch : 2.11.0+cu126
CUDA    : True  (  NVIDIA GeForce RTX 3090)
Device  : cuda

Paths:
  ✓  ATT&CK STIX    OSRs/ATTACK/enterprise-attack-v16.1.json  (40.8 MB)
  ✓  KEV gold       OSRs/kev-07.28.2025_attack-16.1-enterprise.json  (1.2 MB)
  ✓  SMET xlsx      CVE_annotated_dataset.xlsx  (0.1 MB)

✓ Ready for Cell 2.


In [2]:
# Cell 2 — ATT&CK Label Space + Technique Text Builder

with open(CFG.attack_json, encoding="utf-8") as f:
    stix_bundle = json.load(f)

objects    = stix_bundle.get("objects", [])
stix_by_id = {o["id"]: o for o in objects}

# ── 1. Helper: get T-code from STIX object ────────────────────────
def get_tcode(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id", "")
    return ""

# ── 2. All attack-patterns ────────────────────────────────────────
all_techniques  = {}   # tcode → stix obj
technique_names = {}   # tcode → name
technique_descs = {}   # tcode → STIX description text

for o in objects:
    if o.get("type") != "attack-pattern":
        continue
    tc = get_tcode(o)
    if not tc.startswith("T"):
        continue
    all_techniques[tc]  = o
    technique_names[tc] = o.get("name", "")
    technique_descs[tc] = o.get("description", "")

print(f"Total attack-patterns (incl. subs): {len(all_techniques)}")

# ── 3. parent_map, revoked_map ────────────────────────────────────
parent_map  = {}   # sub → parent tcode
revoked_map = {}   # old → new tcode

for o in objects:
    if o.get("type") != "relationship":
        continue
    rt = o.get("relationship_type", "")
    src = stix_by_id.get(o.get("source_ref", ""))
    tgt = stix_by_id.get(o.get("target_ref", ""))
    if not src or not tgt:
        continue
    tc_src = get_tcode(src)
    tc_tgt = get_tcode(tgt)
    if rt == "subtechnique-of" and tc_src and tc_tgt:
        parent_map[tc_src] = tc_tgt
    elif rt == "revoked-by" and tc_src and tc_tgt:
        revoked_map[tc_src] = tc_tgt

print(f"Sub→parent mappings : {len(parent_map)}")
print(f"Revoked mappings    : {len(revoked_map)}")

def resolve_to_parent(tc):
    tc = revoked_map.get(tc, tc)
    tc = parent_map.get(tc, tc)
    tc = revoked_map.get(tc, tc)
    return tc

# ── 4. Active parent techniques (no dot, not revoked) ─────────────
revoked_tcodes = set(revoked_map.keys())
for tc, obj in all_techniques.items():
    if obj.get("x_mitre_revoked", False) or obj.get("revoked", False):
        revoked_tcodes.add(tc)

parent_techniques = {
    tc: obj for tc, obj in all_techniques.items()
    if "." not in tc and tc not in revoked_tcodes
}
print(f"Active parent techniques: {len(parent_techniques)}")

# ── 5. Sub-technique names grouped by parent ──────────────────────
sub_names_by_parent = defaultdict(list)   # parent tcode → [sub names]
for tc, obj in all_techniques.items():
    if "." not in tc:
        continue
    parent_tc = parent_map.get(tc)
    if parent_tc and parent_tc in parent_techniques:
        sub_names_by_parent[parent_tc].append(obj.get("name", ""))

print(f"Parents with sub-techniques: "
      f"{sum(1 for v in sub_names_by_parent.values() if v)}")

# ── 6. Build technique text for encoding ─────────────────────────
# Format: "[T1190] Exploit Public-Facing Application.
#          <STIX description>.
#          Sub-techniques: SQLi, PHP injection, ..."
def build_technique_text(tc):
    name  = technique_names.get(tc, "")
    desc  = technique_descs.get(tc, "")
    # Clean STIX markdown citations like (Citation: ...)
    desc  = re.sub(r'\(Citation:[^)]+\)', '', desc).strip()
    # Truncate description to first 3 sentences to stay within token budget
    sents = re.split(r'(?<=[.!?])\s+', desc)
    desc_short = " ".join(sents[:3]).strip()
    subs  = sub_names_by_parent.get(tc, [])
    text  = f"[{tc}] {name}."
    if desc_short:
        text += f" {desc_short}"
    if subs:
        text += f" Sub-techniques: {', '.join(subs)}."
    return text

# Build index: sorted list of all 214 parent T-codes
TECHNIQUE_LIST  = sorted(parent_techniques.keys())
TECHNIQUE_INDEX = {tc: i for i, tc in enumerate(TECHNIQUE_LIST)}
NUM_TECHNIQUES  = len(TECHNIQUE_LIST)

technique_texts = [build_technique_text(tc) for tc in TECHNIQUE_LIST]

print(f"\nTechnique index size: {NUM_TECHNIQUES}")
print(f"\nSample technique texts:")
for tc in ["T1190", "T1059", "T1566", "T1078"]:
    if tc in TECHNIQUE_INDEX:
        txt = technique_texts[TECHNIQUE_INDEX[tc]]
        print(f"\n  [{tc}] {txt[:200]}{'...' if len(txt)>200 else ''}")

# Token length distribution
print(f"\nTechnique text lengths (chars):")
lens = [len(t) for t in technique_texts]
print(f"  min={min(lens)}  max={max(lens)}  "
      f"mean={np.mean(lens):.0f}  median={np.median(lens):.0f}")

print("\n✓ Label space built. Ready for Cell 3.")

Total attack-patterns (incl. subs): 799
Sub→parent mappings : 456
Revoked mappings    : 139
Active parent techniques: 214
Parents with sub-techniques: 96

Technique index size: 214

Sample technique texts:

  [T1190] [T1190] Exploit Public-Facing Application. Adversaries may attempt to exploit a weakness in an Internet-facing host or system to initially access a network. The weakness in the system can be a softwar...

  [T1059] [T1059] Command and Scripting Interpreter. Adversaries may abuse command and script interpreters to execute commands, scripts, or binaries. These interfaces and languages provide ways of interacting w...

  [T1566] [T1566] Phishing. Adversaries may send phishing messages to gain access to victim systems. All forms of phishing are electronically delivered social engineering. Phishing can be targeted, known as spe...

  [T1078] [T1078] Valid Accounts. Adversaries may obtain and abuse credentials of existing accounts as a means of gaining Initial Access, Persistenc

# Zero-shot on SecureBert+

In [14]:
# Cell 3 — Load Encoder & Build Technique Index

# ── Fix double-prefix in technique texts ─────────────────────────
def build_technique_text(tc):
    name = technique_names.get(tc, "")
    desc = technique_descs.get(tc, "")
    desc = re.sub(r'\(Citation:[^)]+\)', '', desc).strip()
    sents = re.split(r'(?<=[.!?])\s+', desc)
    desc_short = " ".join(sents[:3]).strip()
    subs = sub_names_by_parent.get(tc, [])
    text = f"[{tc}] {name}."
    if desc_short:
        text += f" {desc_short}"
    if subs:
        text += f" Sub-techniques: {', '.join(subs)}."
    return text

# Rebuild cleanly
technique_texts = [build_technique_text(tc) for tc in TECHNIQUE_LIST]

# Verify fix
sample = technique_texts[TECHNIQUE_INDEX["T1190"]]
print(f"Fixed sample: {sample[:120]}")

# ── 1. Load tokenizer + encoder ───────────────────────────────────
print("\nLoading SecureBERT_Plus...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(CFG.encoder_name)
encoder   = AutoModel.from_pretrained(CFG.encoder_name).to(DEVICE)
encoder.eval()

total_params = sum(p.numel() for p in encoder.parameters())
print(f"✓ Encoder loaded  params={total_params/1e6:.1f}M  "
      f"vocab={tokenizer.vocab_size:,}")

# ── 2. Encode helper ──────────────────────────────────────────────
@torch.no_grad()
def encode_texts(texts, batch_size=32, desc="Encoding"):
    """
    Tokenize and encode a list of texts.
    Returns L2-normalised CLS embeddings: (N, 768) float32 numpy array.
    """
    all_embs = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start: start + batch_size]
        enc = tokenizer(
            batch_texts,
            max_length=CFG.max_len,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)
        with autocast():
            out = encoder(input_ids=input_ids,
                          attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # (B, 768)
        cls = F.normalize(cls.float(), p=2, dim=-1)
        all_embs.append(cls.cpu().numpy())
        if (start // batch_size + 1) % 10 == 0 or \
                start + batch_size >= len(texts):
            print(f"\r  {desc}: {min(start+batch_size, len(texts))}"
                  f"/{len(texts)}", end="", flush=True)
    print()
    return np.vstack(all_embs)   # (N, 768)

# ── 3. Encode all 214 technique texts ────────────────────────────
print("\nBuilding technique index...")
technique_embs = encode_texts(
    technique_texts,
    batch_size=32,
    desc="Techniques",
)   # (214, 768)  L2-normalised

print(f"Technique index shape : {technique_embs.shape}")
print(f"Norm check (should≈1) : "
      f"min={np.linalg.norm(technique_embs, axis=1).min():.4f}  "
      f"max={np.linalg.norm(technique_embs, axis=1).max():.4f}")

# ── 4. Similarity retrieval function ─────────────────────────────
def retrieve_top_k(query_embs, k=10):
    """
    query_embs : (N, 768) L2-normalised
    Returns    : scores (N, 214), indices (N, 214) sorted desc
    """
    # cosine sim = dot product of L2-normalised vectors
    sim = query_embs @ technique_embs.T          # (N, 214)
    ranked_idx   = np.argsort(-sim, axis=1)      # (N, 214) desc
    ranked_scores = np.take_along_axis(sim, ranked_idx, axis=1)
    return ranked_scores, ranked_idx

# ── 5. Build full similarity matrix helper for metrics ────────────
def get_score_matrix(query_embs):
    """Returns (N, NUM_TECHNIQUES) cosine similarity matrix."""
    return query_embs @ technique_embs.T         # (N, 214)

# ── 6. Recall@K helper ────────────────────────────────────────────
def recall_at_k(labels, scores, k):
    """
    labels : (N, C) binary
    scores : (N, C) similarity scores
    Returns macro-averaged R@K.
    """
    top_k = np.argsort(-scores, axis=1)[:, :k]
    hits, total = 0, 0
    for i in range(len(labels)):
        pos = set(np.where(labels[i])[0])
        if not pos:
            continue
        hits  += len(pos & set(top_k[i]))
        total += len(pos)
    return hits / total if total > 0 else 0.0

# ── 7. Quick sanity check — encode 3 test CVEs ───────────────────
test_cves = [
    "SQL injection vulnerability allows remote attacker to execute "
    "arbitrary SQL commands via the user input field.",
    "Buffer overflow in the FTP server allows remote code execution "
    "via a long USER command.",
    "Phishing campaign uses spoofed emails to steal credentials from "
    "corporate users.",
]
print("\nSanity check — top-3 techniques per test CVE:")
test_embs = encode_texts(test_cves, batch_size=8, desc="Test CVEs")
_, top_idx = retrieve_top_k(test_embs, k=3)
for i, cve_text in enumerate(test_cves):
    print(f"\n  CVE: {cve_text[:70]}...")
    for rank, idx in enumerate(top_idx[i]):
        tc   = TECHNIQUE_LIST[idx]
        name = technique_names[tc]
        print(f"    #{rank+1}  {tc}  {name}")

print("\n✓ Technique index ready. Ready for Cell 4.")

Fixed sample: [T1190] Exploit Public-Facing Application. Adversaries may attempt to exploit a weakness in an Internet-facing host or s

Loading SecureBERT_Plus...


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 8287.55it/s]


✓ Encoder loaded  params=149.0M  vocab=50,280

Building technique index...
  Techniques: 214/214
Technique index shape : (214, 768)
Norm check (should≈1) : min=1.0000  max=1.0000

Sanity check — top-3 techniques per test CVE:
  Test CVEs: 3/3

  CVE: SQL injection vulnerability allows remote attacker to execute arbitrar...
    #1  T1220  XSL Script Processing
    #2  T1538  Cloud Service Dashboard
    #3  T1056  Input Capture
    #4  T1037  Boot or Logon Initialization Scripts
    #5  T1053  Scheduled Task/Job
    #6  T1137  Office Application Startup
    #7  T1203  Exploitation for Client Execution
    #8  T1221  Template Injection
    #9  T1539  Steal Web Session Cookie
    #10  T1613  Container and Resource Discovery
    #11  T1609  Container Administration Command
    #12  T1505  Server Software Component
    #13  T1567  Exfiltration Over Web Service
    #14  T1651  Cloud Administration Command
    #15  T1127  Trusted Developer Utilities Proxy Execution
    #16  T1047  Windows Mana

In [15]:
# Cell 4 — KEV + SMET Evaluation

# ── Shared metrics helpers ────────────────────────────────────────
def recall_at_k(labels, scores, k):
    top_k = np.argsort(-scores, axis=1)[:, :k]
    hits, total = 0, 0
    for i in range(len(labels)):
        pos = set(np.where(labels[i])[0])
        if not pos:
            continue
        hits  += len(pos & set(top_k[i]))
        total += len(pos)
    return hits / total if total > 0 else 0.0

def full_metrics(scores, labels, tag, k_list=(1,3,5,10)):
    lrap = label_ranking_average_precision_score(labels, scores)
    rl   = label_ranking_loss(labels, scores)
    ce   = coverage_error(labels, scores)
    print(f"\n── {tag} ──────────────────────────────────────────")
    print(f"  LRAP           : {lrap:.4f}")
    print(f"  Ranking Loss   : {rl:.4f}")
    print(f"  Coverage Error : {ce:.4f}")
    for k in k_list:
        rk = recall_at_k(labels, scores, k)
        print(f"  R@{k:<3}          : {rk:.4f}")
    return dict(lrap=lrap, ranking_loss=rl, coverage_error=ce,
                **{f"r{k}": recall_at_k(labels, scores, k)
                   for k in k_list})

# ══════════════════════════════════════════════════════════════════
# A. KEV EVALUATION
# ══════════════════════════════════════════════════════════════════

# ── 1. Load & parse KEV ───────────────────────────────────────────
with open(CFG.kev_json, encoding="utf-8") as f:
    kev_data = json.load(f)
kev_entries = kev_data["mapping_objects"]

# Group by CVE → set of parent T-codes
kev_by_cve = defaultdict(set)
kev_desc   = {}
for entry in kev_entries:
    cve_id = entry.get("capability_id", "").strip()
    tc_raw = entry.get("attack_object_id", "").strip()
    desc   = entry.get("capability_description", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    parent_tc = resolve_to_parent(tc_raw)
    if parent_tc in TECHNIQUE_INDEX:
        kev_by_cve[cve_id].add(parent_tc)
    if desc and cve_id not in kev_desc:
        kev_desc[cve_id] = desc

# Build eval dataframe
kev_rows = [{"cve_id": cid, "description": kev_desc.get(cid, ""),
             "techniques": list(techs)}
            for cid, techs in kev_by_cve.items() if techs]
df_kev = pd.DataFrame(kev_rows)
print(f"KEV eval rows       : {len(df_kev)}")
print(f"Unique T-codes      : "
      f"{len(set(t for ts in df_kev['techniques'] for t in ts))}")

# Drop rows with empty description
df_kev = df_kev[df_kev["description"].str.len() > 10].reset_index(drop=True)
print(f"Rows with desc      : {len(df_kev)}")

# ── 2. Encode KEV descriptions ────────────────────────────────────
print("\nEncoding KEV descriptions...", flush=True)
kev_embs   = encode_texts(df_kev["description"].tolist(),
                           batch_size=CFG.batch_size, desc="KEV")
kev_scores = get_score_matrix(kev_embs)          # (N, 214)

# ── 3. Build label matrix ─────────────────────────────────────────
mlb_full = MultiLabelBinarizer(classes=TECHNIQUE_LIST)
mlb_full.fit([TECHNIQUE_LIST])
kev_labels = mlb_full.transform(
    df_kev["techniques"].tolist()).astype(np.float32)   # (N, 214)

print(f"kev_scores shape    : {kev_scores.shape}")
print(f"kev_labels shape    : {kev_labels.shape}")
print(f"label density       : {kev_labels.mean():.4f}")

# ── 4. KEV metrics ────────────────────────────────────────────────
r_kev = full_metrics(kev_scores, kev_labels,
                     "KEV zero-shot (n={})".format(len(df_kev)))

# ══════════════════════════════════════════════════════════════════
# B. SMET EVALUATION
# ══════════════════════════════════════════════════════════════════

# ── 5. Load SMET ──────────────────────────────────────────────────
df_smet = pd.read_excel(CFG.smet_xlsx, engine="openpyxl")

# ── 6. Build name→T-code map ──────────────────────────────────────
name_to_tcode = {}
for tc, obj in parent_techniques.items():
    name_to_tcode[obj.get("name","").strip().lower()] = tc
for tc, name in technique_names.items():
    parent_tc = resolve_to_parent(tc)
    if parent_tc in TECHNIQUE_INDEX:
        name_to_tcode[name.strip().lower()] = parent_tc

# id2mitre fallback
try:
    import urllib.request
    with urllib.request.urlopen(CFG.smet_id2mitre_url, timeout=10) as r:
        id2mitre = json.loads(r.read().decode("utf-8"))
    for k, v in id2mitre.items():
        kl = k.strip().lower()
        if isinstance(v, str) and re.match(r'T\d{4}', v):
            parent_tc = resolve_to_parent(v)
            if parent_tc in TECHNIQUE_INDEX:
                name_to_tcode[kl] = parent_tc
    print(f"\nid2mitre loaded: {len(id2mitre)} entries")
except Exception as e:
    print(f"\nid2mitre fetch failed: {e}")

print(f"name→T-code entries : {len(name_to_tcode)}")

def parse_smet_techniques(val):
    if pd.isna(val):
        return []
    try:
        names = ast.literal_eval(str(val))
    except Exception:
        names = [str(val)]
    return [name_to_tcode[n.strip().lower()]
            for n in names if n.strip().lower() in name_to_tcode]

df_smet["tcodes"] = df_smet["ATT&CK Techniques"].apply(parse_smet_techniques)
df_smet_eval = df_smet[df_smet["tcodes"].apply(len) > 0].reset_index(drop=True)

print(f"SMET rows mapped    : {len(df_smet_eval)} / {len(df_smet)}")
unique_smet_tc = set(t for ts in df_smet_eval["tcodes"] for t in ts)
print(f"Unique T-codes      : {len(unique_smet_tc)}  {sorted(unique_smet_tc)}")

# ── 7. Encode SMET descriptions ───────────────────────────────────
print("\nEncoding SMET descriptions...", flush=True)
smet_embs   = encode_texts(df_smet_eval["Description"].tolist(),
                            batch_size=CFG.batch_size, desc="SMET")
smet_scores = get_score_matrix(smet_embs)        # (N, 214)

smet_labels = mlb_full.transform(
    df_smet_eval["tcodes"].tolist()).astype(np.float32)

print(f"smet_scores shape   : {smet_scores.shape}")
print(f"smet_labels shape   : {smet_labels.shape}")
print(f"label density       : {smet_labels.mean():.4f}")

# ── 8. SMET metrics ───────────────────────────────────────────────
r_smet = full_metrics(smet_scores, smet_labels,
                      "SMET zero-shot (n={})  [paper: CE=13.96 RL=0.05 "
                      "LRAP=53.77% R@5=67.71%]".format(len(df_smet_eval)))

# ══════════════════════════════════════════════════════════════════
# C. SUMMARY
# ══════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("ZERO-SHOT SUMMARY")
print("═"*65)
print(f"{'Eval Set':<30} {'LRAP':>7} {'RL':>7} {'CE':>7} "
      f"{'R@1':>6} {'R@5':>6} {'R@10':>6}")
print("─"*65)
for tag, r in [("KEV (n={})".format(len(df_kev)),   r_kev),
               ("SMET (n={})".format(len(df_smet_eval)), r_smet),
               ("SMET paper baseline",
                dict(lrap=0.5377, ranking_loss=0.05,
                     coverage_error=13.96, r1=None, r5=0.6771, r10=None))]:
    lrap = f"{r['lrap']:.4f}"
    rl   = f"{r['ranking_loss']:.4f}"
    ce   = f"{r['coverage_error']:.4f}"
    r1   = f"{r['r1']:.4f}" if r.get('r1') is not None else "  —  "
    r5   = f"{r['r5']:.4f}" if r.get('r5') is not None else "  —  "
    r10  = f"{r.get('r10',None):.4f}" \
           if r.get('r10') is not None else "  —  "
    print(f"{tag:<30} {lrap:>7} {rl:>7} {ce:>7} "
          f"{r1:>6} {r5:>6} {r10:>6}")
print("═"*65)

print("\n✓ Zero-shot evaluation complete.")

KEV eval rows       : 419
Unique T-codes      : 108
Rows with desc      : 419

Encoding KEV descriptions...
  KEV: 419/419
kev_scores shape    : (419, 214)
kev_labels shape    : (419, 214)
label density       : 0.0130

── KEV zero-shot (n=419) ──────────────────────────────────────────
  LRAP           : 0.0305
  Ranking Loss   : 0.5074
  Coverage Error : 152.0453
  R@1            : 0.0009
  R@3            : 0.0077
  R@5            : 0.0162
  R@10           : 0.0470

id2mitre loaded: 594 entries
name→T-code entries : 670
SMET rows mapped    : 302 / 303
Unique T-codes      : 40  ['T1005', 'T1007', 'T1016', 'T1040', 'T1055', 'T1059', 'T1068', 'T1078', 'T1083', 'T1110', 'T1136', 'T1176', 'T1189', 'T1190', 'T1195', 'T1203', 'T1204', 'T1211', 'T1213', 'T1485', 'T1491', 'T1498', 'T1499', 'T1505', 'T1518', 'T1528', 'T1529', 'T1531', 'T1539', 'T1543', 'T1547', 'T1548', 'T1552', 'T1557', 'T1562', 'T1565', 'T1566', 'T1574', 'T1598', 'T1606']

Encoding SMET descriptions...
  SMET: 302/302
smet_sc

In [16]:
import json
import pandas as pd
import ast
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# ==========================================
# 1. Parse MITRE ATT&CK (Build attack_dict)
# ==========================================
print("Loading ATT&CK STIX data...")
with open("OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

attack_dict = {}
name_to_tcode = {} # Useful fallback for SMET parsing
parent_map = {}

# First pass: Extract all attack patterns
for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                t_code = ref.get("external_id")
                break
        
        if t_code:
            # We use the raw STIX description. 
            # (Optional: you could strip HTML/Markdown tags here if desired)
            attack_dict[t_code] = obj.get("description", "")
            name_to_tcode[obj.get("name")] = t_code

# Second pass: Build parent map for sub-techniques (optional for this specific test, but good for alignment)
for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id = obj.get("source_ref")
        parent_id = obj.get("target_ref")
        
        # Resolve STIX IDs to T-Codes
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        
        if sub_tcode and parent_tcode:
            parent_map[sub_tcode] = parent_tcode

print(f"Loaded {len(attack_dict)} total techniques/sub-techniques from ATT&CK v16.1")

# ==========================================
# 2. Parse SMET Benchmark (Build smet_df)
# ==========================================
print("Loading SMET benchmark data...")
smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")

with open("id2mitre.json", "r", encoding="utf-8") as f:
    id2mitre = json.load(f)

smet_records = []
for _, row in smet_raw.iterrows():
    cve_id = row["ID"]
    description = row["Description"]
    
    # SMET stores techniques as a string representation of a list of names
    try:
        tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except:
        tech_names = []
        
    t_codes = []
    for name in tech_names:
        # 1. Try mapping via SMET's provided JSON
        mapped_id = None
        for key, vals in id2mitre.items():
            if name in vals:
                mapped_id = key
                break
        
        # 2. Fallback to STIX exact name match
        if not mapped_id:
            mapped_id = name_to_tcode.get(name)
            
        if mapped_id:
            # For this test, you can either keep sub-techniques or roll them up to parents.
            # We will roll up to match your earlier parent-only label space rule.
            final_id = parent_map.get(mapped_id, mapped_id)
            t_codes.append(final_id)
            
    if t_codes:
        # Deduplicate T-codes after parent rollup
        smet_records.append({
            "CVE_ID": cve_id,
            "Description": description,
            "T_Codes": list(set(t_codes))
        })

smet_df = pd.DataFrame(smet_records)
print(f"Successfully parsed {len(smet_df)} SMET CVEs with valid ATT&CK mappings.")

# ==========================================
# 3. Setup Model and Device
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "cisco-ai/SecureBERT2.0-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

# ==========================================
# 4. Define Mean-Pooling Function
# ==========================================
def get_embeddings(text_list, batch_size=16):
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size), desc="Embedding"):
            batch_texts = text_list[i:i+batch_size]
            
            encoded_input = tokenizer(
                batch_texts, 
                padding=True, 
                truncation=True, 
                max_length=512, 
                return_tensors='pt'
            ).to(device)
            
            model_output = model(**encoded_input)
            
            attention_mask = encoded_input['attention_mask']
            token_embeddings = model_output.last_hidden_state
            
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            
            mean_pooled = sum_embeddings / sum_mask
            mean_pooled = F.normalize(mean_pooled, p=2, dim=1)
            
            all_embeddings.append(mean_pooled.cpu())
            
    return torch.cat(all_embeddings, dim=0)

# ==========================================
# 5. Execute Zero-Shot Evaluation
# ==========================================
# Filter attack_dict to only include PARENT techniques (no ".") to match your thesis constraints
parent_attack_dict = {k: v for k, v in attack_dict.items() if "." not in k}
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

print(f"\nEmbedding {len(technique_texts)} Parent ATT&CK Techniques...")
tech_embeddings = get_embeddings(technique_texts, batch_size=16)

print(f"\nEmbedding {len(cve_texts)} SMET CVEs...")
cve_embeddings = get_embeddings(cve_texts, batch_size=16)

# Compute similarity matrix
similarity_matrix = torch.matmul(cve_embeddings, tech_embeddings.T)

hits_at_1 = 0
hits_at_5 = 0
hits_at_10 = 0

for i in range(len(cve_texts)):
    true_labels = set(cve_ground_truths[i])
    
    # Get top 10 indices
    top_10_indices = torch.topk(similarity_matrix[i], k=10).indices.tolist()
    top_10_predictions = [technique_ids[idx] for idx in top_10_indices]
    
    # R@1
    if top_10_predictions[0] in true_labels:
        hits_at_1 += 1
        
    # R@5
    if len(true_labels.intersection(set(top_10_predictions[:5]))) > 0:
        hits_at_5 += 1
        
    # R@10
    if len(true_labels.intersection(set(top_10_predictions))) > 0:
        hits_at_10 += 1

n_cves = len(cve_texts)
print("\n" + "="*50)
print("ZERO-SHOT BI-ENCODER RESULTS (Untrained)")
print("="*50)
print(f"Total SMET CVEs Evaluated: {n_cves}")
print(f"Candidate Techniques (Parents Only): {len(technique_ids)}")
print("-" * 50)
print(f"Recall@1:  {(hits_at_1 / n_cves) * 100:.2f}%")
print(f"Recall@5:  {(hits_at_5 / n_cves) * 100:.2f}%")
print(f"Recall@10: {(hits_at_10 / n_cves) * 100:.2f}%")
print("="*50)

Loading ATT&CK STIX data...
Loaded 656 total techniques/sub-techniques from ATT&CK v16.1
Loading SMET benchmark data...
Successfully parsed 302 SMET CVEs with valid ATT&CK mappings.
Using device: cuda


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 18447.39it/s]



Embedding 203 Parent ATT&CK Techniques...


Embedding: 100%|██████████| 13/13 [00:02<00:00,  6.48it/s]



Embedding 302 SMET CVEs...


Embedding: 100%|██████████| 19/19 [00:01<00:00, 16.99it/s]


ZERO-SHOT BI-ENCODER RESULTS (Untrained)
Total SMET CVEs Evaluated: 302
Candidate Techniques (Parents Only): 203
--------------------------------------------------
Recall@1:  5.30%
Recall@5:  12.91%
Recall@10: 18.87%


# Running on AttackBert

In [3]:
# Cell 3 — Load Encoder & Build Technique Index

# ── Fix double-prefix in technique texts ─────────────────────────
def build_technique_text(tc):
    name = technique_names.get(tc, "")
    desc = technique_descs.get(tc, "")
    desc = re.sub(r'\(Citation:[^)]+\)', '', desc).strip()
    sents = re.split(r'(?<=[.!?])\s+', desc)
    desc_short = " ".join(sents[:3]).strip()
    subs = sub_names_by_parent.get(tc, [])
    text = f"[{tc}] {name}."
    if desc_short:
        text += f" {desc_short}"
    if subs:
        text += f" Sub-techniques: {', '.join(subs)}."
    return text

# Rebuild cleanly
technique_texts = [build_technique_text(tc) for tc in TECHNIQUE_LIST]

# Verify fix
sample = technique_texts[TECHNIQUE_INDEX["T1190"]]
print(f"Fixed sample: {sample[:120]}")

# ── 1. Load tokenizer + encoder ───────────────────────────────────
print("\nLoading AttackBer...", flush=True)
tokenizer = AutoTokenizer.from_pretrained("basel/ATTACK-BERT")
encoder   = AutoModel.from_pretrained("basel/ATTACK-BERT").to(DEVICE)
encoder.eval()

total_params = sum(p.numel() for p in encoder.parameters())
print(f"✓ Encoder loaded  params={total_params/1e6:.1f}M  "
      f"vocab={tokenizer.vocab_size:,}")

# ── 2. Encode helper ──────────────────────────────────────────────
@torch.no_grad()
def encode_texts(texts, batch_size=32, desc="Encoding"):
    """
    Tokenize and encode a list of texts.
    Returns L2-normalised CLS embeddings: (N, 768) float32 numpy array.
    """
    all_embs = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start: start + batch_size]
        enc = tokenizer(
            batch_texts,
            max_length=CFG.max_len,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)
        with autocast():
            out = encoder(input_ids=input_ids,
                          attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # (B, 768)
        cls = F.normalize(cls.float(), p=2, dim=-1)
        all_embs.append(cls.cpu().numpy())
        if (start // batch_size + 1) % 10 == 0 or \
                start + batch_size >= len(texts):
            print(f"\r  {desc}: {min(start+batch_size, len(texts))}"
                  f"/{len(texts)}", end="", flush=True)
    print()
    return np.vstack(all_embs)   # (N, 768)

# ── 3. Encode all 214 technique texts ────────────────────────────
print("\nBuilding technique index...")
technique_embs = encode_texts(
    technique_texts,
    batch_size=32,
    desc="Techniques",
)   # (214, 768)  L2-normalised

print(f"Technique index shape : {technique_embs.shape}")
print(f"Norm check (should≈1) : "
      f"min={np.linalg.norm(technique_embs, axis=1).min():.4f}  "
      f"max={np.linalg.norm(technique_embs, axis=1).max():.4f}")

# ── 4. Similarity retrieval function ─────────────────────────────
def retrieve_top_k(query_embs, k=10):
    """
    query_embs : (N, 768) L2-normalised
    Returns    : scores (N, 214), indices (N, 214) sorted desc
    """
    # cosine sim = dot product of L2-normalised vectors
    sim = query_embs @ technique_embs.T          # (N, 214)
    ranked_idx   = np.argsort(-sim, axis=1)      # (N, 214) desc
    ranked_scores = np.take_along_axis(sim, ranked_idx, axis=1)
    return ranked_scores, ranked_idx

# ── 5. Build full similarity matrix helper for metrics ────────────
def get_score_matrix(query_embs):
    """Returns (N, NUM_TECHNIQUES) cosine similarity matrix."""
    return query_embs @ technique_embs.T         # (N, 214)

# ── 6. Recall@K helper ────────────────────────────────────────────
def recall_at_k(labels, scores, k):
    """
    labels : (N, C) binary
    scores : (N, C) similarity scores
    Returns macro-averaged R@K.
    """
    top_k = np.argsort(-scores, axis=1)[:, :k]
    hits, total = 0, 0
    for i in range(len(labels)):
        pos = set(np.where(labels[i])[0])
        if not pos:
            continue
        hits  += len(pos & set(top_k[i]))
        total += len(pos)
    return hits / total if total > 0 else 0.0

# ── 7. Quick sanity check — encode 3 test CVEs ───────────────────
test_cves = [
    "SQL injection vulnerability allows remote attacker to execute "
    "arbitrary SQL commands via the user input field.",
    "Buffer overflow in the FTP server allows remote code execution "
    "via a long USER command.",
    "Phishing campaign uses spoofed emails to steal credentials from "
    "corporate users.",
]
print("\nSanity check — top-3 techniques per test CVE:")
test_embs = encode_texts(test_cves, batch_size=8, desc="Test CVEs")
_, top_idx = retrieve_top_k(test_embs, k=3)
for i, cve_text in enumerate(test_cves):
    print(f"\n  CVE: {cve_text[:70]}...")
    for rank, idx in enumerate(top_idx[i]):
        tc   = TECHNIQUE_LIST[idx]
        name = technique_names[tc]
        print(f"    #{rank+1}  {tc}  {name}")

print("\n✓ Technique index ready. Ready for Cell 4.")

Fixed sample: [T1190] Exploit Public-Facing Application. Adversaries may attempt to exploit a weakness in an Internet-facing host or s

Loading AttackBer...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 48420.15it/s]


✓ Encoder loaded  params=109.5M  vocab=30,527

Building technique index...
  Techniques: 214/214
Technique index shape : (214, 768)
Norm check (should≈1) : min=1.0000  max=1.0000

Sanity check — top-3 techniques per test CVE:
  Test CVEs: 3/3

  CVE: SQL injection vulnerability allows remote attacker to execute arbitrar...
    #1  T1659  Content Injection
    #2  T1190  Exploit Public-Facing Application
    #3  T1059  Command and Scripting Interpreter
    #4  T1212  Exploitation for Credential Access
    #5  T1202  Indirect Command Execution
    #6  T1203  Exploitation for Client Execution
    #7  T1064  Scripting
    #8  T1189  Drive-by Compromise
    #9  T1221  Template Injection
    #10  T1211  Exploitation for Defense Evasion
    #11  T1565  Data Manipulation
    #12  T1584  Compromise Infrastructure
    #13  T1055  Process Injection
    #14  T1204  User Execution
    #15  T1068  Exploitation for Privilege Escalation
    #16  T1586  Compromise Accounts
    #17  T1554  Compromise Ho

In [4]:
# Cell 4 — KEV + SMET Evaluation

# ── Shared metrics helpers ────────────────────────────────────────
def recall_at_k(labels, scores, k):
    top_k = np.argsort(-scores, axis=1)[:, :k]
    hits, total = 0, 0
    for i in range(len(labels)):
        pos = set(np.where(labels[i])[0])
        if not pos:
            continue
        hits  += len(pos & set(top_k[i]))
        total += len(pos)
    return hits / total if total > 0 else 0.0

def full_metrics(scores, labels, tag, k_list=(1,3,5,10)):
    lrap = label_ranking_average_precision_score(labels, scores)
    rl   = label_ranking_loss(labels, scores)
    ce   = coverage_error(labels, scores)
    print(f"\n── {tag} ──────────────────────────────────────────")
    print(f"  LRAP           : {lrap:.4f}")
    print(f"  Ranking Loss   : {rl:.4f}")
    print(f"  Coverage Error : {ce:.4f}")
    for k in k_list:
        rk = recall_at_k(labels, scores, k)
        print(f"  R@{k:<3}          : {rk:.4f}")
    return dict(lrap=lrap, ranking_loss=rl, coverage_error=ce,
                **{f"r{k}": recall_at_k(labels, scores, k)
                   for k in k_list})

# ══════════════════════════════════════════════════════════════════
# A. KEV EVALUATION
# ══════════════════════════════════════════════════════════════════

# ── 1. Load & parse KEV ───────────────────────────────────────────
with open(CFG.kev_json, encoding="utf-8") as f:
    kev_data = json.load(f)
kev_entries = kev_data["mapping_objects"]

# Group by CVE → set of parent T-codes
kev_by_cve = defaultdict(set)
kev_desc   = {}
for entry in kev_entries:
    cve_id = entry.get("capability_id", "").strip()
    tc_raw = entry.get("attack_object_id", "").strip()
    desc   = entry.get("capability_description", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    parent_tc = resolve_to_parent(tc_raw)
    if parent_tc in TECHNIQUE_INDEX:
        kev_by_cve[cve_id].add(parent_tc)
    if desc and cve_id not in kev_desc:
        kev_desc[cve_id] = desc

# Build eval dataframe
kev_rows = [{"cve_id": cid, "description": kev_desc.get(cid, ""),
             "techniques": list(techs)}
            for cid, techs in kev_by_cve.items() if techs]
df_kev = pd.DataFrame(kev_rows)
print(f"KEV eval rows       : {len(df_kev)}")
print(f"Unique T-codes      : "
      f"{len(set(t for ts in df_kev['techniques'] for t in ts))}")

# Drop rows with empty description
df_kev = df_kev[df_kev["description"].str.len() > 10].reset_index(drop=True)
print(f"Rows with desc      : {len(df_kev)}")

# ── 2. Encode KEV descriptions ────────────────────────────────────
print("\nEncoding KEV descriptions...", flush=True)
kev_embs   = encode_texts(df_kev["description"].tolist(),
                           batch_size=CFG.batch_size, desc="KEV")
kev_scores = get_score_matrix(kev_embs)          # (N, 214)

# ── 3. Build label matrix ─────────────────────────────────────────
mlb_full = MultiLabelBinarizer(classes=TECHNIQUE_LIST)
mlb_full.fit([TECHNIQUE_LIST])
kev_labels = mlb_full.transform(
    df_kev["techniques"].tolist()).astype(np.float32)   # (N, 214)

print(f"kev_scores shape    : {kev_scores.shape}")
print(f"kev_labels shape    : {kev_labels.shape}")
print(f"label density       : {kev_labels.mean():.4f}")

# ── 4. KEV metrics ────────────────────────────────────────────────
r_kev = full_metrics(kev_scores, kev_labels,
                     "KEV zero-shot (n={})".format(len(df_kev)))

# ══════════════════════════════════════════════════════════════════
# B. SMET EVALUATION
# ══════════════════════════════════════════════════════════════════

# ── 5. Load SMET ──────────────────────────────────────────────────
df_smet = pd.read_excel(CFG.smet_xlsx, engine="openpyxl")

# ── 6. Build name→T-code map ──────────────────────────────────────
name_to_tcode = {}
for tc, obj in parent_techniques.items():
    name_to_tcode[obj.get("name","").strip().lower()] = tc
for tc, name in technique_names.items():
    parent_tc = resolve_to_parent(tc)
    if parent_tc in TECHNIQUE_INDEX:
        name_to_tcode[name.strip().lower()] = parent_tc

# id2mitre fallback
try:
    import urllib.request
    with urllib.request.urlopen(CFG.smet_id2mitre_url, timeout=10) as r:
        id2mitre = json.loads(r.read().decode("utf-8"))
    for k, v in id2mitre.items():
        kl = k.strip().lower()
        if isinstance(v, str) and re.match(r'T\d{4}', v):
            parent_tc = resolve_to_parent(v)
            if parent_tc in TECHNIQUE_INDEX:
                name_to_tcode[kl] = parent_tc
    print(f"\nid2mitre loaded: {len(id2mitre)} entries")
except Exception as e:
    print(f"\nid2mitre fetch failed: {e}")

print(f"name→T-code entries : {len(name_to_tcode)}")

def parse_smet_techniques(val):
    if pd.isna(val):
        return []
    try:
        names = ast.literal_eval(str(val))
    except Exception:
        names = [str(val)]
    return [name_to_tcode[n.strip().lower()]
            for n in names if n.strip().lower() in name_to_tcode]

df_smet["tcodes"] = df_smet["ATT&CK Techniques"].apply(parse_smet_techniques)
df_smet_eval = df_smet[df_smet["tcodes"].apply(len) > 0].reset_index(drop=True)

print(f"SMET rows mapped    : {len(df_smet_eval)} / {len(df_smet)}")
unique_smet_tc = set(t for ts in df_smet_eval["tcodes"] for t in ts)
print(f"Unique T-codes      : {len(unique_smet_tc)}  {sorted(unique_smet_tc)}")

# ── 7. Encode SMET descriptions ───────────────────────────────────
print("\nEncoding SMET descriptions...", flush=True)
smet_embs   = encode_texts(df_smet_eval["Description"].tolist(),
                            batch_size=CFG.batch_size, desc="SMET")
smet_scores = get_score_matrix(smet_embs)        # (N, 214)

smet_labels = mlb_full.transform(
    df_smet_eval["tcodes"].tolist()).astype(np.float32)

print(f"smet_scores shape   : {smet_scores.shape}")
print(f"smet_labels shape   : {smet_labels.shape}")
print(f"label density       : {smet_labels.mean():.4f}")

# ── 8. SMET metrics ───────────────────────────────────────────────
r_smet = full_metrics(smet_scores, smet_labels,
                      "SMET zero-shot (n={})  [paper: CE=13.96 RL=0.05 "
                      "LRAP=53.77% R@5=67.71%]".format(len(df_smet_eval)))

# ══════════════════════════════════════════════════════════════════
# C. SUMMARY
# ══════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("ZERO-SHOT SUMMARY")
print("═"*65)
print(f"{'Eval Set':<30} {'LRAP':>7} {'RL':>7} {'CE':>7} "
      f"{'R@1':>6} {'R@5':>6} {'R@10':>6}")
print("─"*65)
for tag, r in [("KEV (n={})".format(len(df_kev)),   r_kev),
               ("SMET (n={})".format(len(df_smet_eval)), r_smet),
               ("SMET paper baseline",
                dict(lrap=0.5377, ranking_loss=0.05,
                     coverage_error=13.96, r1=None, r5=0.6771, r10=None))]:
    lrap = f"{r['lrap']:.4f}"
    rl   = f"{r['ranking_loss']:.4f}"
    ce   = f"{r['coverage_error']:.4f}"
    r1   = f"{r['r1']:.4f}" if r.get('r1') is not None else "  —  "
    r5   = f"{r['r5']:.4f}" if r.get('r5') is not None else "  —  "
    r10  = f"{r.get('r10',None):.4f}" \
           if r.get('r10') is not None else "  —  "
    print(f"{tag:<30} {lrap:>7} {rl:>7} {ce:>7} "
          f"{r1:>6} {r5:>6} {r10:>6}")
print("═"*65)

print("\n✓ Zero-shot evaluation complete.")

KEV eval rows       : 419
Unique T-codes      : 108
Rows with desc      : 419

Encoding KEV descriptions...
  KEV: 419/419
kev_scores shape    : (419, 214)
kev_labels shape    : (419, 214)
label density       : 0.0130

── KEV zero-shot (n=419) ──────────────────────────────────────────
  LRAP           : 0.1063
  Ranking Loss   : 0.2815
  Coverage Error : 101.5274
  R@1            : 0.0179
  R@3            : 0.0607
  R@5            : 0.0957
  R@10           : 0.1726

id2mitre loaded: 594 entries
name→T-code entries : 670
SMET rows mapped    : 302 / 303
Unique T-codes      : 40  ['T1005', 'T1007', 'T1016', 'T1040', 'T1055', 'T1059', 'T1068', 'T1078', 'T1083', 'T1110', 'T1136', 'T1176', 'T1189', 'T1190', 'T1195', 'T1203', 'T1204', 'T1211', 'T1213', 'T1485', 'T1491', 'T1498', 'T1499', 'T1505', 'T1518', 'T1528', 'T1529', 'T1531', 'T1539', 'T1543', 'T1547', 'T1548', 'T1552', 'T1557', 'T1562', 'T1565', 'T1566', 'T1574', 'T1598', 'T1606']

Encoding SMET descriptions...
  SMET: 302/302
smet_sc

In [5]:
import json
import pandas as pd
import ast
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# ==========================================
# 1. Parse MITRE ATT&CK (Build attack_dict)
# ==========================================
print("Loading ATT&CK STIX data...")
with open("OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

attack_dict = {}
name_to_tcode = {} # Useful fallback for SMET parsing
parent_map = {}

# First pass: Extract all attack patterns
for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                t_code = ref.get("external_id")
                break
        
        if t_code:
            # We use the raw STIX description. 
            # (Optional: you could strip HTML/Markdown tags here if desired)
            attack_dict[t_code] = obj.get("description", "")
            name_to_tcode[obj.get("name")] = t_code

# Second pass: Build parent map for sub-techniques (optional for this specific test, but good for alignment)
for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id = obj.get("source_ref")
        parent_id = obj.get("target_ref")
        
        # Resolve STIX IDs to T-Codes
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        
        if sub_tcode and parent_tcode:
            parent_map[sub_tcode] = parent_tcode

print(f"Loaded {len(attack_dict)} total techniques/sub-techniques from ATT&CK v16.1")

# ==========================================
# 2. Parse SMET Benchmark (Build smet_df)
# ==========================================
print("Loading SMET benchmark data...")
smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")

with open("id2mitre.json", "r", encoding="utf-8") as f:
    id2mitre = json.load(f)

smet_records = []
for _, row in smet_raw.iterrows():
    cve_id = row["ID"]
    description = row["Description"]
    
    # SMET stores techniques as a string representation of a list of names
    try:
        tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except:
        tech_names = []
        
    t_codes = []
    for name in tech_names:
        # 1. Try mapping via SMET's provided JSON
        mapped_id = None
        for key, vals in id2mitre.items():
            if name in vals:
                mapped_id = key
                break
        
        # 2. Fallback to STIX exact name match
        if not mapped_id:
            mapped_id = name_to_tcode.get(name)
            
        if mapped_id:
            # For this test, you can either keep sub-techniques or roll them up to parents.
            # We will roll up to match your earlier parent-only label space rule.
            final_id = parent_map.get(mapped_id, mapped_id)
            t_codes.append(final_id)
            
    if t_codes:
        # Deduplicate T-codes after parent rollup
        smet_records.append({
            "CVE_ID": cve_id,
            "Description": description,
            "T_Codes": list(set(t_codes))
        })

smet_df = pd.DataFrame(smet_records)
print(f"Successfully parsed {len(smet_df)} SMET CVEs with valid ATT&CK mappings.")

# ==========================================
# 3. Setup Model and Device
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "basel/ATTACK-BERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

# ==========================================
# 4. Define Mean-Pooling Function
# ==========================================
def get_embeddings(text_list, batch_size=16):
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size), desc="Embedding"):
            batch_texts = text_list[i:i+batch_size]
            
            encoded_input = tokenizer(
                batch_texts, 
                padding=True, 
                truncation=True, 
                max_length=512, 
                return_tensors='pt'
            ).to(device)
            
            model_output = model(**encoded_input)
            
            attention_mask = encoded_input['attention_mask']
            token_embeddings = model_output.last_hidden_state
            
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            
            mean_pooled = sum_embeddings / sum_mask
            mean_pooled = F.normalize(mean_pooled, p=2, dim=1)
            
            all_embeddings.append(mean_pooled.cpu())
            
    return torch.cat(all_embeddings, dim=0)

# ==========================================
# 5. Execute Zero-Shot Evaluation
# ==========================================
# Filter attack_dict to only include PARENT techniques (no ".") to match your thesis constraints
parent_attack_dict = {k: v for k, v in attack_dict.items() if "." not in k}
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

print(f"\nEmbedding {len(technique_texts)} Parent ATT&CK Techniques...")
tech_embeddings = get_embeddings(technique_texts, batch_size=16)

print(f"\nEmbedding {len(cve_texts)} SMET CVEs...")
cve_embeddings = get_embeddings(cve_texts, batch_size=16)

# Compute similarity matrix
similarity_matrix = torch.matmul(cve_embeddings, tech_embeddings.T)

hits_at_1 = 0
hits_at_5 = 0
hits_at_10 = 0

for i in range(len(cve_texts)):
    true_labels = set(cve_ground_truths[i])
    
    # Get top 10 indices
    top_10_indices = torch.topk(similarity_matrix[i], k=10).indices.tolist()
    top_10_predictions = [technique_ids[idx] for idx in top_10_indices]
    
    # R@1
    if top_10_predictions[0] in true_labels:
        hits_at_1 += 1
        
    # R@5
    if len(true_labels.intersection(set(top_10_predictions[:5]))) > 0:
        hits_at_5 += 1
        
    # R@10
    if len(true_labels.intersection(set(top_10_predictions))) > 0:
        hits_at_10 += 1

n_cves = len(cve_texts)
print("\n" + "="*50)
print("ZERO-SHOT BI-ENCODER RESULTS (Untrained)")
print("="*50)
print(f"Total SMET CVEs Evaluated: {n_cves}")
print(f"Candidate Techniques (Parents Only): {len(technique_ids)}")
print("-" * 50)
print(f"Recall@1:  {(hits_at_1 / n_cves) * 100:.2f}%")
print(f"Recall@5:  {(hits_at_5 / n_cves) * 100:.2f}%")
print(f"Recall@10: {(hits_at_10 / n_cves) * 100:.2f}%")
print("="*50)

Loading ATT&CK STIX data...
Loaded 656 total techniques/sub-techniques from ATT&CK v16.1
Loading SMET benchmark data...
Successfully parsed 302 SMET CVEs with valid ATT&CK mappings.
Using device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24812.46it/s]



Embedding 203 Parent ATT&CK Techniques...


Embedding: 100%|██████████| 13/13 [00:01<00:00,  7.91it/s]



Embedding 302 SMET CVEs...


Embedding: 100%|██████████| 19/19 [00:00<00:00, 23.17it/s]


ZERO-SHOT BI-ENCODER RESULTS (Untrained)
Total SMET CVEs Evaluated: 302
Candidate Techniques (Parents Only): 203
--------------------------------------------------
Recall@1:  25.17%
Recall@5:  55.30%
Recall@10: 68.54%


In [6]:
from rank_bm25 import BM25Okapi
import numpy as np
import torch

# ==========================================
# 1. Build the BM25 Lexical Index
# (Run this right after you build your technique_texts list)
# ==========================================
print("\nBuilding BM25 Lexical Index...")
# Simple whitespace tokenizer for BM25
tokenized_corpus = [text.lower().split() for text in technique_texts]
bm25_model = BM25Okapi(tokenized_corpus)

# ==========================================
# 2. The Smart Hybrid RRF Function
# ==========================================
def hybrid_rrf_search(cve_text, semantic_scores_tensor, bm25_model, technique_ids, top_k=10, rrf_k=60):
    """
    Combines Semantic (ATT&CK-BERT) and Lexical (BM25) search using Reciprocal Rank Fusion.
    """
    # 1. Get Semantic Ranks (from the pre-computed similarity matrix)
    # Convert tensor to numpy for easier ranking
    semantic_scores = semantic_scores_tensor.cpu().numpy()
    
    # argsort gives ascending order, so we reverse it [::-1] for descending (highest score first)
    semantic_ranked_indices = np.argsort(semantic_scores)[::-1]
    
    # Create a dictionary mapping the index to its Semantic Rank (1st place = rank 1)
    semantic_ranks = {idx: rank + 1 for rank, idx in enumerate(semantic_ranked_indices)}
    
    # 2. Get Lexical Ranks (BM25)
    tokenized_query = cve_text.lower().split()
    bm25_scores = bm25_model.get_scores(tokenized_query)
    
    lexical_ranked_indices = np.argsort(bm25_scores)[::-1]
    lexical_ranks = {idx: rank + 1 for rank, idx in enumerate(lexical_ranked_indices)}
    
    # 3. Compute RRF Scores
    rrf_scores = {}
    for idx in range(len(technique_ids)):
        # Apply the RRF formula
        s_rank = semantic_ranks[idx]
        l_rank = lexical_ranks[idx]
        
        # If BM25 score is exactly 0.0 (no keyword overlap at all), penalize its rank to infinity
        if bm25_scores[idx] == 0.0:
            l_rank = float('inf')
            
        rrf_score = (1.0 / (rrf_k + s_rank)) + (1.0 / (rrf_k + l_rank))
        rrf_scores[idx] = rrf_score
        
    # 4. Sort by final RRF Score
    sorted_rrf_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    
    # Return the top K T-codes
    top_predictions = [technique_ids[idx] for idx in sorted_rrf_indices[:top_k]]
    return top_predictions

# ==========================================
# 3. Execute Hybrid Zero-Shot Evaluation
# (Replace your existing eval loop with this)
# ==========================================
print("\n" + "="*50)
print("Evaluating Smart Hybrid Search (ATT&CK-BERT + BM25 + RRF)")
print("="*50)

# Compute full semantic similarity matrix once (just like you did before)
similarity_matrix = torch.matmul(cve_embeddings, tech_embeddings.T)

hits_at_1 = 0
hits_at_5 = 0
hits_at_10 = 0

for i in tqdm(range(len(cve_texts)), desc="Hybrid Inference"):
    true_labels = set(cve_ground_truths[i])
    current_cve_text = cve_texts[i]
    
    # Get predictions using the smart RRF function
    top_10_predictions = hybrid_rrf_search(
        cve_text=current_cve_text,
        semantic_scores_tensor=similarity_matrix[i],
        bm25_model=bm25_model,
        technique_ids=technique_ids,
        top_k=10,
        rrf_k=60 # 60 is the industry standard constant for RRF
    )
    
    # R@1
    if top_10_predictions[0] in true_labels:
        hits_at_1 += 1
        
    # R@5
    if len(true_labels.intersection(set(top_10_predictions[:5]))) > 0:
        hits_at_5 += 1
        
    # R@10
    if len(true_labels.intersection(set(top_10_predictions))) > 0:
        hits_at_10 += 1

n_cves = len(cve_texts)
print("\n" + "="*50)
print("SMART HYBRID RESULTS (Zero-Shot)")
print("="*50)
print(f"Total SMET CVEs Evaluated: {n_cves}")
print("-" * 50)
print(f"Hybrid Hit Rate@1:  {(hits_at_1 / n_cves) * 100:.2f}%")
print(f"Hybrid Hit Rate@5:  {(hits_at_5 / n_cves) * 100:.2f}%")
print(f"Hybrid Hit Rate@10: {(hits_at_10 / n_cves) * 100:.2f}%")
print("="*50)


Building BM25 Lexical Index...

Evaluating Smart Hybrid Search (ATT&CK-BERT + BM25 + RRF)


Hybrid Inference: 100%|██████████| 302/302 [00:00<00:00, 635.58it/s]


SMART HYBRID RESULTS (Zero-Shot)
Total SMET CVEs Evaluated: 302
--------------------------------------------------
Hybrid Hit Rate@1:  32.45%
Hybrid Hit Rate@5:  63.91%
Hybrid Hit Rate@10: 76.16%


In [38]:
# ==========================================
# 6. KEV Gold Evaluation (same model, same index)
# ==========================================
print("Loading KEV gold labels...")
with open("OSRs/kev-07.28.2025_attack-16.1-enterprise.json", "r", encoding="utf-8") as f:
    kev_data = json.load(f)

# Group by CVE → (description, set of parent T-codes)
kev_by_cve = defaultdict(lambda: {"desc": "", "techs": set()})
for entry in kev_data["mapping_objects"]:
    cve_id  = entry.get("capability_id", "").strip()
    tc_raw  = entry.get("attack_object_id", "").strip()
    desc    = entry.get("capability_description", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    # Roll sub-techniques up to parent (same logic as SMET)
    parent_tc = parent_map.get(tc_raw, tc_raw)
    # Only keep parent techniques (no dot) that exist in our index
    if "." not in parent_tc and parent_tc in parent_attack_dict:
        kev_by_cve[cve_id]["techs"].add(parent_tc)
    if desc:
        kev_by_cve[cve_id]["desc"] = desc

# Build eval lists — drop CVEs with empty desc or no mapped techniques
kev_cve_ids, kev_texts, kev_ground_truths = [], [], []
for cve_id, val in kev_by_cve.items():
    if val["desc"] and val["techs"]:
        kev_cve_ids.append(cve_id)
        kev_texts.append(val["desc"])
        kev_ground_truths.append(val["techs"])

print(f"KEV CVEs with desc + mapped techniques : {len(kev_texts)}")
print(f"Techniques outside parent index (dropped): "
      f"{sum(1 for e in kev_data['mapping_objects'] if '.' in e.get('attack_object_id',''))}"
      f" sub-technique rows")

# ── Embed KEV descriptions (technique index already built above) ──
print(f"\nEmbedding {len(kev_texts)} KEV CVE descriptions...")
kev_embeddings = get_embeddings(kev_texts, batch_size=16)

# Cosine similarity against same technique index
kev_sim = torch.matmul(kev_embeddings, tech_embeddings.T)

# ── Metrics (R@1, R@5, R@10 — same calculation as SMET) ──────────
kev_hits_1 = kev_hits_5 = kev_hits_10 = 0

for i in range(len(kev_texts)):
    true_labels  = kev_ground_truths[i]
    top_10_idx   = torch.topk(kev_sim[i], k=10).indices.tolist()
    top_10_preds = [technique_ids[idx] for idx in top_10_idx]

    if top_10_preds[0] in true_labels:
        kev_hits_1 += 1
    if true_labels.intersection(set(top_10_preds[:5])):
        kev_hits_5 += 1
    if true_labels.intersection(set(top_10_preds)):
        kev_hits_10 += 1

n_kev = len(kev_texts)
print("\n" + "="*50)
print("ZERO-SHOT BI-ENCODER RESULTS — KEV GOLD")
print("="*50)
print(f"Total KEV CVEs Evaluated    : {n_kev}")
print(f"Candidate Techniques        : {len(technique_ids)} parents")
print("-"*50)
print(f"Recall@1  : {kev_hits_1/n_kev*100:.2f}%")
print(f"Recall@5  : {kev_hits_5/n_kev*100:.2f}%")
print(f"Recall@10 : {kev_hits_10/n_kev*100:.2f}%")
print("="*50)

# ── Side-by-side comparison ───────────────────────────────────────
print("\n" + "="*50)
print("COMPARISON: SMET vs KEV (AttackBERT zero-shot)")
print("="*50)
print(f"{'Metric':<12} {'SMET':>10} {'KEV':>10}")
print("-"*35)
print(f"{'R@1':<12} {hits_at_1/n_cves*100:>9.2f}% {kev_hits_1/n_kev*100:>9.2f}%")
print(f"{'R@5':<12} {hits_at_5/n_cves*100:>9.2f}% {kev_hits_5/n_kev*100:>9.2f}%")
print(f"{'R@10':<12} {hits_at_10/n_cves*100:>9.2f}% {kev_hits_10/n_kev*100:>9.2f}%")
print(f"{'N':<12} {n_cves:>10} {n_kev:>10}")
print("="*50)

Loading KEV gold labels...
KEV CVEs with desc + mapped techniques : 419
Techniques outside parent index (dropped): 222 sub-technique rows

Embedding 419 KEV CVE descriptions...


Embedding: 100%|██████████| 27/27 [00:00<00:00, 45.25it/s]


ZERO-SHOT BI-ENCODER RESULTS — KEV GOLD
Total KEV CVEs Evaluated    : 419
Candidate Techniques        : 203 parents
--------------------------------------------------
Recall@1  : 8.11%
Recall@5  : 26.73%
Recall@10 : 41.05%

COMPARISON: SMET vs KEV (AttackBERT zero-shot)
Metric             SMET        KEV
-----------------------------------
R@1              32.45%      8.11%
R@5              63.91%     26.73%
R@10             76.16%     41.05%
N                   302        419


In [44]:
# ==========================================
# 4. Hybrid Evaluation on KEV Gold
# ==========================================
print("\nEvaluating Hybrid Search on KEV...")
kev_sim = torch.matmul(kev_embeddings, tech_embeddings.T)

kev_h1 = kev_h5 = kev_h10 = 0
for i in tqdm(range(len(kev_texts)), desc="KEV Hybrid"):
    true_labels = kev_ground_truths[i]
    top_10 = hybrid_rrf_search(
        cve_text=kev_texts[i],
        semantic_scores_tensor=kev_sim[i],
        bm25_model=bm25_model,
        technique_ids=technique_ids,
        top_k=10,
        rrf_k=60,
    )
    if top_10[0] in true_labels:                         kev_h1  += 1
    if true_labels.intersection(set(top_10[:5])):        kev_h5  += 1
    if true_labels.intersection(set(top_10)):            kev_h10 += 1

n_kev = len(kev_texts)
print("\n" + "="*55)
print("FINAL COMPARISON — Semantic vs Hybrid (Zero-Shot)")
print("="*55)
print(f"{'Metric':<12} {'SMET Sem':>10} {'SMET Hyb':>10} "
      f"{'KEV Sem':>10} {'KEV Hyb':>10}")
print("-"*55)
for metric, s_sem, s_hyb, k_sem, k_hyb in [
    ("R@1",  hits_at_1,  None,    kev_hits_1,  kev_h1),
    ("R@5",  hits_at_5,  None,    kev_hits_5,  kev_h5),
    ("R@10", hits_at_10, None,    kev_hits_10, kev_h10),
]:
    # pure semantic from your earlier run
    s_sem_pct = s_sem / n_cves * 100
    k_sem_pct = k_sem / n_kev * 100
    k_hyb_pct = k_hyb / n_kev * 100
    print(f"{metric:<12}{'(above)':>10} {s_sem_pct:>9.2f}%  "
          f"{k_sem_pct:>9.2f}% {k_hyb_pct:>9.2f}%")
print(f"{'N':<12} {n_cves:>10} {'':>10} {n_kev:>10}")
print("="*55)


Evaluating Hybrid Search on KEV...


KEV Hybrid: 100%|██████████| 419/419 [00:00<00:00, 2064.28it/s]


FINAL COMPARISON — Semantic vs Hybrid (Zero-Shot)
Metric         SMET Sem   SMET Hyb    KEV Sem    KEV Hyb
-------------------------------------------------------
R@1            (above)     32.45%       8.11%     12.89%
R@5            (above)     63.91%      26.73%     26.25%
R@10           (above)     76.16%      41.05%     40.57%
N                   302                   419


# STOP HERE the rest are experiments

In [12]:
# Cell A — Enrich Technique Texts with Mitigations + Examples

# ── Build mitigation text per technique from STIX ─────────────────
# Mitigations linked via "mitigates" relationships
tech_mitigations = defaultdict(list)   # tcode → [mitigation descriptions]
tech_detections  = defaultdict(list)   # tcode → [detection text]

# Index mitigations by STIX id
mitigations_by_id = {}
for obj in stix_bundle["objects"]:
    if obj.get("type") == "course-of-action":
        mitigations_by_id[obj["id"]] = obj.get("description", "")

# Follow "mitigates" relationships
for obj in stix_bundle["objects"]:
    if obj.get("type") != "relationship":
        continue
    if obj.get("relationship_type") != "mitigates":
        continue
    tgt_id   = obj.get("target_ref", "")
    src_id   = obj.get("source_ref", "")
    tgt_obj  = next((o for o in stix_bundle["objects"]
                     if o.get("id") == tgt_id), None)
    if not tgt_obj:
        continue
    tc = next((r["external_id"] for r in
               tgt_obj.get("external_references", [])
               if r.get("source_name") == "mitre-attack"), None)
    if tc and src_id in mitigations_by_id:
        desc = mitigations_by_id[src_id]
        if desc:
            tech_mitigations[tc].append(desc)

# Extract x_mitre_detection field (detection guidance)
for obj in stix_bundle["objects"]:
    if obj.get("type") != "attack-pattern":
        continue
    tc = next((r["external_id"] for r in
               obj.get("external_references", [])
               if r.get("source_name") == "mitre-attack"), None)
    det = obj.get("x_mitre_detection", "")
    if tc and det:
        tech_detections[tc].append(det)

print(f"Techniques with mitigation text : "
      f"{sum(1 for v in tech_mitigations.values() if v)}")
print(f"Techniques with detection text  : "
      f"{sum(1 for v in tech_detections.values() if v)}")

# ── Rebuild technique texts ────────────────────────────────────────
def build_enriched_technique_text(tc, max_mit_chars=300, max_det_chars=300):
    name  = name_to_tcode.get(tc, tc)   # use tcode as fallback
    # get name properly
    tname = next((obj.get("name","") for obj in stix_bundle["objects"]
                  if obj.get("type") == "attack-pattern"
                  and any(r.get("external_id") == tc
                          for r in obj.get("external_references", [])
                          if r.get("source_name") == "mitre-attack")), tc)
    desc  = attack_dict.get(tc, "")
    # clean citations
    desc  = re.sub(r'\(Citation:[^)]+\)', '', desc).strip()
    # first 3 sentences
    sents = re.split(r'(?<=[.!?])\s+', desc)
    desc_short = " ".join(sents[:3]).strip()

    # sub-technique names
    subs = [obj.get("name","") for obj in stix_bundle["objects"]
            if obj.get("type") == "attack-pattern"
            and any(r.get("external_id","").startswith(tc + ".")
                    for r in obj.get("external_references", [])
                    if r.get("source_name") == "mitre-attack")]

    # mitigation summary (truncated)
    mit_text = " ".join(tech_mitigations.get(tc, []))
    mit_text = re.sub(r'\(Citation:[^)]+\)', '', mit_text)[:max_mit_chars].strip()

    # detection summary (truncated)
    det_text = " ".join(tech_detections.get(tc, []))
    det_text = re.sub(r'\(Citation:[^)]+\)', '', det_text)[:max_det_chars].strip()

    parts = [f"[{tc}] {tname}."]
    if desc_short:
        parts.append(desc_short)
    if subs:
        parts.append(f"Sub-techniques: {', '.join(subs[:8])}.")
    if mit_text:
        parts.append(f"Mitigations: {mit_text}")
    if det_text:
        parts.append(f"Detection: {det_text}")
    return " ".join(parts)

import re
from collections import defaultdict

# Rebuild index texts
technique_texts_enriched = [
    build_enriched_technique_text(tc) for tc in technique_ids]

# Compare lengths
orig_len  = np.mean([len(t) for t in technique_texts])
enrich_len = np.mean([len(t) for t in technique_texts_enriched])
print(f"\nAvg technique text length:")
print(f"  Before enrichment : {orig_len:.0f} chars")
print(f"  After enrichment  : {enrich_len:.0f} chars")
print(f"\nSample (T1190):")
idx_t1190 = technique_ids.index("T1190")
print(technique_texts_enriched[idx_t1190][:400])

# ── Re-embed technique index ───────────────────────────────────────
print("\nRe-embedding enriched technique index...")
tech_embeddings_enriched = get_embeddings(
    technique_texts_enriched, batch_size=16)
tech_embeddings_enriched_np = tech_embeddings_enriched.numpy()

# ── Rebuild BM25 on enriched texts ────────────────────────────────
print("Rebuilding BM25 on enriched texts...")
tokenized_corpus_enriched = [t.lower().split()
                              for t in technique_texts_enriched]
bm25_enriched = BM25Okapi(tokenized_corpus_enriched)
print("✓ Enriched index ready.")

Techniques with mitigation text : 729
Techniques with detection text  : 725

Avg technique text length:
  Before enrichment : 1301 chars
  After enrichment  : 1098 chars

Sample (T1190):
[T1190] Exploit Public-Facing Application. Adversaries may attempt to exploit a weakness in an Internet-facing host or system to initially access a network. The weakness in the system can be a software bug, a temporary glitch, or a misconfiguration. Exploited applications are often websites/web servers, but can also include databases (like SQL), standard services (like SMB or SSH), network device 

Re-embedding enriched technique index...


Embedding: 100%|██████████| 13/13 [00:01<00:00, 11.26it/s]

Rebuilding BM25 on enriched texts...
✓ Enriched index ready.


In [13]:
# Cell B — Enrich KEV Descriptions from NVD 2.0 + Re-evaluate

import glob, json

# ── 1. Build CVE-ID → full NVD description lookup ─────────────────
print("Building NVD lookup (this takes ~30s)...")
nvd_desc_lookup = {}   # "CVE-2021-1234" → full English description

nvd_files = sorted(glob.glob("OSRs/NVD/nvdcve-2.0-*.json"))
print(f"  NVD files found: {len(nvd_files)}")

for fpath in nvd_files:
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)
    for entry in data.get("vulnerabilities", []):
        cve_obj = entry.get("cve", {})
        cve_id  = cve_obj.get("id", "")
        for d in cve_obj.get("descriptions", []):
            if d.get("lang") == "en":
                txt = d.get("value", "").strip()
                if txt and not txt.startswith("** REJECT"):
                    nvd_desc_lookup[cve_id] = txt
                break

print(f"  NVD descriptions loaded: {len(nvd_desc_lookup):,}")

# ── 2. Enrich KEV descriptions ────────────────────────────────────
kev_texts_enriched = []
enriched_count = 0
for cve_id, text in zip(kev_cve_ids, kev_texts):
    nvd_text = nvd_desc_lookup.get(cve_id, "")
    if nvd_text and len(nvd_text) > len(text):
        # Prepend short KEV desc + full NVD desc
        kev_texts_enriched.append(f"{text} {nvd_text}")
        enriched_count += 1
    else:
        kev_texts_enriched.append(text)

print(f"\nKEV descriptions enriched via NVD : {enriched_count} / {len(kev_texts)}")
avg_before = np.mean([len(t) for t in kev_texts])
avg_after  = np.mean([len(t) for t in kev_texts_enriched])
print(f"Avg desc length before : {avg_before:.0f} chars")
print(f"Avg desc length after  : {avg_after:.0f} chars")

# ── 3. Re-embed enriched KEV descriptions ────────────────────────
print("\nRe-embedding enriched KEV descriptions...")
kev_embs_enriched = get_embeddings(kev_texts_enriched, batch_size=16)

# ── 4. Evaluate: 4 combinations ──────────────────────────────────
configs = [
    ("Semantic  | orig  tech + orig  KEV",
     kev_embeddings,     tech_embeddings,          technique_texts,
     bm25_model,         kev_texts),
    ("Semantic  | enr   tech + enr   KEV",
     kev_embs_enriched,  tech_embeddings_enriched, technique_texts_enriched,
     bm25_enriched,      kev_texts_enriched),
    ("Hybrid RRF| orig  tech + orig  KEV",
     kev_embeddings,     tech_embeddings,          technique_texts,
     bm25_model,         kev_texts),
    ("Hybrid RRF| enr   tech + enr   KEV",
     kev_embs_enriched,  tech_embeddings_enriched, technique_texts_enriched,
     bm25_enriched,      kev_texts_enriched),
]

use_hybrid = [False, False, True, True]

print("\n" + "="*65)
print("KEV ABLATION — Enrichment Impact")
print("="*65)
print(f"{'Config':<42} {'R@1':>7} {'R@5':>7} {'R@10':>7}")
print("─"*65)

for (label, q_emb, t_emb, t_texts, bm25_idx, q_texts), hybrid in \
        zip(configs, use_hybrid):
    sim = torch.matmul(q_emb, t_emb.T)
    h1 = h5 = h10 = 0
    for i in range(len(kev_texts)):
        true_labels = kev_ground_truths[i]
        if hybrid:
            top10 = hybrid_rrf_search(
                q_texts[i], sim[i], bm25_idx, technique_ids, top_k=10)
        else:
            top10 = [technique_ids[j]
                     for j in torch.topk(sim[i], 10).indices.tolist()]
        if top10[0] in true_labels:               h1  += 1
        if true_labels.intersection(top10[:5]):   h5  += 1
        if true_labels.intersection(top10):       h10 += 1
    n = len(kev_texts)
    print(f"{label:<42} {h1/n*100:>6.2f}% {h5/n*100:>6.2f}% {h10/n*100:>6.2f}%")

# Also re-run SMET with enriched technique index for comparison
print("─"*65)
sim_smet_enr = torch.matmul(cve_embeddings, tech_embeddings_enriched.T)
s1 = s5 = s10 = 0
for i in range(len(cve_texts)):
    true_labels = set(cve_ground_truths[i])
    top10 = hybrid_rrf_search(
        cve_texts[i], sim_smet_enr[i], bm25_enriched,
        technique_ids, top_k=10)
    if top10[0] in true_labels:                    s1  += 1
    if true_labels.intersection(set(top10[:5])):   s5  += 1
    if true_labels.intersection(set(top10)):       s10 += 1
n_s = len(cve_texts)
print(f"{'SMET Hybrid | enr tech (baseline: 63.91%)':<42} "
      f"{s1/n_s*100:>6.2f}% {s5/n_s*100:>6.2f}% {s10/n_s*100:>6.2f}%")
print("="*65)

Building NVD lookup (this takes ~30s)...
  NVD files found: 25
  NVD descriptions loaded: 333,022

KEV descriptions enriched via NVD : 375 / 419
Avg desc length before : 62 chars
Avg desc length after  : 395 chars

Re-embedding enriched KEV descriptions...


Embedding: 100%|██████████| 27/27 [00:01<00:00, 16.55it/s]



KEV ABLATION — Enrichment Impact
Config                                         R@1     R@5    R@10
─────────────────────────────────────────────────────────────────
Semantic  | orig  tech + orig  KEV           8.11%  26.73%  41.05%
Semantic  | enr   tech + enr   KEV           8.11%  29.83%  45.35%
Hybrid RRF| orig  tech + orig  KEV          12.89%  26.25%  40.57%
Hybrid RRF| enr   tech + enr   KEV          12.89%  28.64%  47.73%
─────────────────────────────────────────────────────────────────
SMET Hybrid | enr tech (baseline: 63.91%)   32.78%  62.58%  73.84%


In [14]:
# Cell C — Cross-Encoder Re-ranking on Top-20 Candidates

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "sentence-transformers"])
from sentence_transformers import CrossEncoder

# ── 1. Load cross-encoder ─────────────────────────────────────────
print("Loading cross-encoder...")
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=str(device),
)
print("✓ cross-encoder/ms-marco-MiniLM-L-6-v2 loaded")

# ── 2. Re-ranking function ────────────────────────────────────────
def rerank_top_k(cve_text, sim_row, technique_ids, technique_texts,
                 cross_encoder, top_k=10, rerank_pool=20):
    """
    1. Bi-encoder top-rerank_pool candidates
    2. Cross-encoder scores each (cve_text, technique_text) pair
    3. Return top_k by cross-encoder score
    """
    # Step 1 — bi-encoder top-20
    pool_indices = torch.topk(sim_row, rerank_pool).indices.tolist()
    pool_tcodes  = [technique_ids[i]  for i in pool_indices]
    pool_texts   = [technique_texts[i] for i in pool_indices]

    # Step 2 — cross-encoder scoring
    pairs  = [(cve_text, t) for t in pool_texts]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)

    # Step 3 — re-rank
    ranked = sorted(zip(pool_tcodes, scores),
                    key=lambda x: x[1], reverse=True)
    return [tc for tc, _ in ranked[:top_k]]

# ── 3. Evaluate: 4 configs (SMET + KEV) × (no rerank + rerank) ───
print("\nEvaluating cross-encoder re-ranking...")
print("(~1-2 min per eval set)\n")

def eval_set(q_texts, ground_truths, sim_matrix,
             t_ids, t_texts, label, rerank=False, hybrid=False,
             bm25_idx=None):
    h1 = h5 = h10 = 0
    for i in tqdm(range(len(q_texts)), desc=label, leave=False):
        true_labels = set(ground_truths[i]) if isinstance(
            ground_truths[i], list) else ground_truths[i]

        if rerank:
            top10 = rerank_top_k(
                q_texts[i], sim_matrix[i],
                t_ids, t_texts, cross_encoder,
                top_k=10, rerank_pool=20)
        elif hybrid:
            top10 = hybrid_rrf_search(
                q_texts[i], sim_matrix[i],
                bm25_idx, t_ids, top_k=10)
        else:
            top10 = [t_ids[j]
                     for j in torch.topk(sim_matrix[i], 10).indices.tolist()]

        if top10[0] in true_labels:                      h1  += 1
        if true_labels.intersection(set(top10[:5])):     h5  += 1
        if true_labels.intersection(set(top10)):         h10 += 1

    n = len(q_texts)
    return h1/n*100, h5/n*100, h10/n*100

# Pre-compute similarity matrices we need
sim_smet_orig = torch.matmul(cve_embeddings,       tech_embeddings.T)
sim_smet_enr  = torch.matmul(cve_embeddings,       tech_embeddings_enriched.T)
sim_kev_orig  = torch.matmul(kev_embeddings,       tech_embeddings.T)
sim_kev_enr   = torch.matmul(kev_embs_enriched,    tech_embeddings_enriched.T)

results = []

# SMET — orig tech, semantic only (current best: R@5=63.91%)
r = eval_set(cve_texts, cve_ground_truths, sim_smet_orig,
             technique_ids, technique_texts,
             "SMET semantic+orig", rerank=False)
results.append(("SMET", "Semantic  | orig tech",  *r))

# SMET — orig tech + cross-encoder rerank
r = eval_set(cve_texts, cve_ground_truths, sim_smet_orig,
             technique_ids, technique_texts,
             "SMET rerank+orig", rerank=True)
results.append(("SMET", "Rerank CE | orig tech",  *r))

# SMET — enriched + cross-encoder rerank
r = eval_set(cve_texts, cve_ground_truths, sim_smet_enr,
             technique_ids, technique_texts_enriched,
             "SMET rerank+enr", rerank=True)
results.append(("SMET", "Rerank CE | enr tech",   *r))

# KEV — best semantic baseline (enriched)
r = eval_set(kev_texts_enriched, kev_ground_truths, sim_kev_enr,
             technique_ids, technique_texts_enriched,
             "KEV semantic+enr", rerank=False)
results.append(("KEV",  "Semantic  | enr  tech",  *r))

# KEV — hybrid + enriched (current best: R@10=47.73%)
r = eval_set(kev_texts_enriched, kev_ground_truths, sim_kev_enr,
             technique_ids, technique_texts_enriched,
             "KEV hybrid+enr", rerank=False, hybrid=True,
             bm25_idx=bm25_enriched)
results.append(("KEV",  "Hybrid RRF| enr  tech",  *r))

# KEV — cross-encoder rerank on enriched
r = eval_set(kev_texts_enriched, kev_ground_truths, sim_kev_enr,
             technique_ids, technique_texts_enriched,
             "KEV rerank+enr", rerank=True)
results.append(("KEV",  "Rerank CE | enr  tech",  *r))

# KEV — hybrid + cross-encoder (bi-enc → hybrid-20 → rerank)
# For this: first hybrid top-20, then cross-encoder re-scores those 20
def hybrid_then_rerank(q_texts, ground_truths, sim_matrix,
                       t_ids, t_texts, bm25_idx, label):
    h1 = h5 = h10 = 0
    for i in tqdm(range(len(q_texts)), desc=label, leave=False):
        true_labels = set(ground_truths[i]) if isinstance(
            ground_truths[i], list) else ground_truths[i]
        # get hybrid top-20
        pool = hybrid_rrf_search(
            q_texts[i], sim_matrix[i], bm25_idx, t_ids, top_k=20)
        # cross-encoder re-score
        pairs  = [(q_texts[i], t_texts[t_ids.index(tc)]) for tc in pool]
        scores = cross_encoder.predict(pairs, show_progress_bar=False)
        ranked = [tc for tc, _ in sorted(zip(pool, scores),
                                         key=lambda x: x[1], reverse=True)]
        top10  = ranked[:10]
        if top10[0] in true_labels:                      h1  += 1
        if true_labels.intersection(set(top10[:5])):     h5  += 1
        if true_labels.intersection(set(top10)):         h10 += 1
    n = len(q_texts)
    return h1/n*100, h5/n*100, h10/n*100

r = hybrid_then_rerank(kev_texts_enriched, kev_ground_truths,
                       sim_kev_enr, technique_ids,
                       technique_texts_enriched, bm25_enriched,
                       "KEV hybrid→rerank")
results.append(("KEV", "Hybrid→Rerank CE | enr", *r))

# ── 4. Print final table ──────────────────────────────────────────
print("\n" + "="*70)
print("FINAL RESULTS — All Configurations")
print("="*70)
print(f"{'Set':<6} {'Config':<28} {'R@1':>8} {'R@5':>8} {'R@10':>8}")
print("─"*70)
prev_set = None
for set_name, config, r1, r5, r10 in results:
    if prev_set and prev_set != set_name:
        print("─"*70)
    print(f"{set_name:<6} {config:<28} {r1:>7.2f}% {r5:>7.2f}% {r10:>7.2f}%")
    prev_set = set_name
print("─"*70)
print(f"{'SMET':<6} {'Paper baseline':<28} {'32.45':>8} {'67.71':>8} {'  —':>8}")
print("="*70)

Loading cross-encoder...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6145.63it/s]


✓ cross-encoder/ms-marco-MiniLM-L-6-v2 loaded

Evaluating cross-encoder re-ranking...
(~1-2 min per eval set)




FINAL RESULTS — All Configurations
Set    Config                            R@1      R@5     R@10
──────────────────────────────────────────────────────────────────────
SMET   Semantic  | orig tech          25.17%   55.30%   68.54%
SMET   Rerank CE | orig tech          24.83%   55.96%   66.89%
SMET   Rerank CE | enr tech           24.50%   62.25%   72.52%
──────────────────────────────────────────────────────────────────────
KEV    Semantic  | enr  tech           8.11%   29.83%   45.35%
KEV    Hybrid RRF| enr  tech          12.89%   28.64%   47.73%
KEV    Rerank CE | enr  tech          13.13%   35.80%   54.65%
KEV    Hybrid→Rerank CE | enr         13.13%   35.56%   57.52%
──────────────────────────────────────────────────────────────────────
SMET   Paper baseline                  32.45    67.71        —


In [15]:
# Cell C (addendum) — Missing SMET configurations

results_smet_extra = []

# SMET — hybrid + orig tech (was never run)
r = eval_set(cve_texts, cve_ground_truths, sim_smet_orig,
             technique_ids, technique_texts,
             "SMET hybrid+orig", rerank=False, hybrid=True,
             bm25_idx=bm25_model)
results_smet_extra.append(("SMET", "Hybrid RRF| orig tech", *r))

# SMET — semantic + enriched (was never run standalone)
r = eval_set(cve_texts, cve_ground_truths, sim_smet_enr,
             technique_ids, technique_texts_enriched,
             "SMET semantic+enr", rerank=False)
results_smet_extra.append(("SMET", "Semantic  | enr  tech", *r))

# SMET — hybrid + enriched
r = eval_set(cve_texts, cve_ground_truths, sim_smet_enr,
             technique_ids, technique_texts_enriched,
             "SMET hybrid+enr", rerank=False, hybrid=True,
             bm25_idx=bm25_enriched)
results_smet_extra.append(("SMET", "Hybrid RRF| enr  tech", *r))

# SMET — rerank on enriched (run before but missing from table)
r = eval_set(cve_texts, cve_ground_truths, sim_smet_enr,
             technique_ids, technique_texts_enriched,
             "SMET rerank+enr", rerank=True)
results_smet_extra.append(("SMET", "Rerank CE | enr  tech", *r))

# SMET — hybrid → rerank, orig tech
r = hybrid_then_rerank(cve_texts, cve_ground_truths,
                       sim_smet_orig, technique_ids,
                       technique_texts, bm25_model,
                       "SMET hybrid→rerank+orig")
results_smet_extra.append(("SMET", "Hybrid→Rerank CE | orig", *r))

# SMET — hybrid → rerank, enriched tech
r = hybrid_then_rerank(cve_texts, cve_ground_truths,
                       sim_smet_enr, technique_ids,
                       technique_texts_enriched, bm25_enriched,
                       "SMET hybrid→rerank+enr")
results_smet_extra.append(("SMET", "Hybrid→Rerank CE | enr", *r))

# ── Print complete unified table ──────────────────────────────────
all_results = [
    # SMET complete
    ("SMET", "Semantic  | orig tech",      25.17, 55.30, 68.54),
    ("SMET", "Semantic  | enr  tech",      *results_smet_extra[1][2:]),
    ("SMET", "Hybrid RRF| orig tech",      *results_smet_extra[0][2:]),
    ("SMET", "Hybrid RRF| enr  tech",      *results_smet_extra[2][2:]),
    ("SMET", "Rerank CE | orig tech",      24.83, 55.96, 66.89),
    ("SMET", "Rerank CE | enr  tech",      *results_smet_extra[3][2:]),
    ("SMET", "Hybrid→Rerank CE | orig",    *results_smet_extra[4][2:]),
    ("SMET", "Hybrid→Rerank CE | enr",     *results_smet_extra[5][2:]),
    # KEV complete (from previous run)
    ("KEV",  "Semantic  | enr  tech",       8.11, 29.83, 45.35),
    ("KEV",  "Hybrid RRF| enr  tech",      12.89, 28.64, 47.73),
    ("KEV",  "Rerank CE | enr  tech",      13.13, 35.80, 54.65),
    ("KEV",  "Hybrid→Rerank CE | enr",     13.13, 35.56, 57.52),
]

print("\n" + "="*70)
print("COMPLETE RESULTS — All Configurations")
print("="*70)
print(f"{'Set':<6} {'Config':<28} {'R@1':>8} {'R@5':>8} {'R@10':>8}")
print("─"*70)
prev_set = None
for set_name, config, r1, r5, r10 in all_results:
    if prev_set and prev_set != set_name:
        print("─"*70)
    # Bold best per set
    print(f"{set_name:<6} {config:<28} {r1:>7.2f}% {r5:>7.2f}% {r10:>7.2f}%")
    prev_set = set_name
print("─"*70)
print(f"{'SMET':<6} {'Paper baseline':<28} {'32.45':>8} {'67.71':>8} {'  —':>8}")
print("="*70)


COMPLETE RESULTS — All Configurations
Set    Config                            R@1      R@5     R@10
──────────────────────────────────────────────────────────────────────
SMET   Semantic  | orig tech          25.17%   55.30%   68.54%
SMET   Semantic  | enr  tech          21.85%   52.98%   66.56%
SMET   Hybrid RRF| orig tech          32.45%   63.91%   76.16%
SMET   Hybrid RRF| enr  tech          32.78%   62.58%   73.84%
SMET   Rerank CE | orig tech          24.83%   55.96%   66.89%
SMET   Rerank CE | enr  tech          24.50%   62.25%   72.52%
SMET   Hybrid→Rerank CE | orig        20.86%   49.67%   65.89%
SMET   Hybrid→Rerank CE | enr         23.18%   61.92%   79.80%
──────────────────────────────────────────────────────────────────────
KEV    Semantic  | enr  tech           8.11%   29.83%   45.35%
KEV    Hybrid RRF| enr  tech          12.89%   28.64%   47.73%
KEV    Rerank CE | enr  tech          13.13%   35.80%   54.65%
KEV    Hybrid→Rerank CE | enr         13.13%   35.56%   57.52%


In [16]:
# Cell D — Download CVE2CAPEC, Build Lookup & Confidence+Mask System

import urllib.request
import os
import json
import glob
from pathlib import Path
from tqdm import tqdm
import torch
import numpy as np

# ── 1. Download CVE2CAPEC JSONL files from GitHub ─────────────────
CVECAPEC_DIR = Path("OSRs/CVE2CAPEC")
CVECAPEC_DIR.mkdir(parents=True, exist_ok=True)

API_URL  = "https://api.github.com/repos/Galeax/CVE2CAPEC/contents/database"
RAW_BASE = "https://raw.githubusercontent.com/Galeax/CVE2CAPEC/main/database"

print("Fetching file list from GitHub...")
try:
    req = urllib.request.Request(API_URL,
          headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=15) as r:
        file_list = json.loads(r.read().decode("utf-8"))
    jsonl_files = [f["name"] for f in file_list
                   if f["name"].endswith(".jsonl")]
    print(f"Found {len(jsonl_files)} JSONL files: {sorted(jsonl_files)[:5]}...")
except Exception as e:
    print(f"API call failed ({e}) — falling back to year-based URLs")
    jsonl_files = [f"{yr}.jsonl" for yr in range(2002, 2026)]

# Download missing files
downloaded = skipped = 0
for fname in sorted(jsonl_files):
    dest = CVECAPEC_DIR / fname
    if dest.exists():
        skipped += 1
        continue
    url = f"{RAW_BASE}/{fname}"
    try:
        urllib.request.urlretrieve(url, dest)
        downloaded += 1
        print(f"  ✓ {fname}", flush=True)
    except Exception as e:
        print(f"  ✗ {fname}  ({e})")

print(f"\nDownloaded: {downloaded}  Already existed: {skipped}")
local_files = sorted(CVECAPEC_DIR.glob("*.jsonl"))
print(f"Total local files: {len(local_files)}")

# ── 2. Load all JSONL into lookup dict ────────────────────────────
# Format: {"CVE-2007-1234": {"CWE":[...], "CAPEC":[...], "TECHNIQUES":[...]}}
print("\nLoading CVE2CAPEC database...")
cve2capec_db = {}   # CVE-ID → {"CWE", "CAPEC", "TECHNIQUES" (parent tcodes)}

for fpath in local_files:
    with open(fpath, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            for cve_id, data in entry.items():
                # Normalize TECHNIQUES: "1134" → "T1134", roll up to parent
                raw_techs = data.get("TECHNIQUES", [])
                parent_tcodes = set()
                for t in raw_techs:
                    t = t.strip()
                    # add T prefix if missing
                    if not t.upper().startswith("T"):
                        t = "T" + t
                    t = t.upper()
                    # roll sub-technique up to parent
                    parent_tc = parent_map.get(t, t)
                    # only keep if in our technique index
                    if parent_tc in set(technique_ids):
                        parent_tcodes.add(parent_tc)
                cve2capec_db[cve_id] = {
                    "CWE"       : data.get("CWE", []),
                    "CAPEC"     : data.get("CAPEC", []),
                    "TECHNIQUES": list(parent_tcodes),
                }

print(f"CVEs in database         : {len(cve2capec_db):,}")
cvs_with_techs = sum(1 for v in cve2capec_db.values() if v["TECHNIQUES"])
print(f"CVEs with ≥1 technique   : {cvs_with_techs:,}")

# Coverage check on eval sets
kev_covered  = sum(1 for cid in kev_cve_ids if cid in cve2capec_db)
smet_covered = sum(1 for cid in smet_df["CVE_ID"] if cid in cve2capec_db)
print(f"KEV  CVEs in DB          : {kev_covered} / {len(kev_cve_ids)}")
print(f"SMET CVEs in DB          : {smet_covered} / {len(smet_df)}")

# ── 3. Confidence & Mask config ────────────────────────────────────
CFG_LOOKUP = dict(
    margin_high     = 0.05,   # top1 - top2 gap → high confidence
    margin_low      = 0.02,   # below this → low confidence
    soft_mask_weight= 0.3,    # multiply non-allowed scores by this
    top_k           = 5,      # how many techniques to return
)

# ── 4. Core lookup function ────────────────────────────────────────
def lookup_cve(cve_id, cve_description,
               sim_scores_tensor,          # (214,) pre-computed
               technique_ids=technique_ids,
               technique_names=name_to_tcode,
               db=cve2capec_db,
               cfg=CFG_LOOKUP):
    """
    Returns a dict with:
      - top_techniques : list of (tcode, score, name)
      - confidence     : 'high' | 'medium' | 'low'
      - margin         : float
      - db_techniques  : list of tcodes from CVE2CAPEC (empty if not found)
      - agreement      : 'agree' | 'partial' | 'disagree' | 'db_only' | 'no_db'
      - message        : human-readable explanation
      - masked         : bool (was BRON mask applied)
    """
    scores = sim_scores_tensor.cpu().numpy().copy()   # (214,)
    db_entry    = db.get(cve_id, {})
    db_techs    = db_entry.get("TECHNIQUES", [])
    db_tech_set = set(db_techs)
    tid_array   = np.array(technique_ids)

    # ── Step 1: compute margin BEFORE masking ─────────────────────
    top2_idx    = np.argsort(-scores)[:2]
    top1_score  = scores[top2_idx[0]]
    top2_score  = scores[top2_idx[1]]
    margin      = float(top1_score - top2_score)

    if margin >= cfg["margin_high"]:
        confidence = "high"
    elif margin >= cfg["margin_low"]:
        confidence = "medium"
    else:
        confidence = "low"

    # ── Step 2: apply BRON mask ───────────────────────────────────
    masked = False
    if db_techs:
        allowed_idx = np.array([i for i, tc in enumerate(technique_ids)
                                 if tc in db_tech_set])
        masked = True
        if confidence == "high":
            # Hard mask — zero out everything not in DB
            hard_scores = np.full_like(scores, -np.inf)
            hard_scores[allowed_idx] = scores[allowed_idx]
            scores = hard_scores
        else:
            # Soft mask — downweight non-allowed
            mask = np.full(len(technique_ids), cfg["soft_mask_weight"])
            mask[allowed_idx] = 1.0
            scores = scores * mask

    # ── Step 3: get top-K after masking ───────────────────────────
    valid_idx = np.where(np.isfinite(scores))[0]
    ranked    = valid_idx[np.argsort(-scores[valid_idx])]
    top_k_idx = ranked[:cfg["top_k"]]

    top_techniques = []
    for idx in top_k_idx:
        tc   = technique_ids[idx]
        sc   = float(sim_scores_tensor.cpu().numpy()[idx])  # original score
        name = next((k for k, v in name_to_tcode.items() if v == tc), tc)
        top_techniques.append((tc, round(sc, 4), name))

    # ── Step 4: agreement logic ───────────────────────────────────
    predicted_set = {tc for tc, _, _ in top_techniques}

    if not db_techs:
        agreement = "no_db"
        if confidence == "high":
            msg = (f"High confidence (margin={margin:.3f}). "
                   f"No CVE2CAPEC entry — showing semantic result.")
        elif confidence == "medium":
            msg = (f"Medium confidence (margin={margin:.3f}). "
                   f"No CVE2CAPEC entry — treat with caution.")
        else:
            msg = (f"Low confidence (margin={margin:.3f}). "
                   f"No CVE2CAPEC entry — result unreliable.")

    elif confidence == "high":
        overlap = predicted_set & db_tech_set
        if overlap:
            agreement = "agree"
            msg = (f"High confidence (margin={margin:.3f}). "
                   f"Semantic top-{cfg['top_k']} agrees with CVE2CAPEC "
                   f"on: {sorted(overlap)}. Hard mask applied.")
        else:
            agreement = "disagree"
            msg = (f"High confidence (margin={margin:.3f}) but "
                   f"CVE2CAPEC suggests different techniques: "
                   f"{sorted(db_tech_set)}. Showing both — review recommended.")
    else:
        overlap = predicted_set & db_tech_set
        if overlap:
            agreement = "partial"
            msg = (f"{confidence.capitalize()} confidence (margin={margin:.3f}). "
                   f"Partial overlap with CVE2CAPEC. "
                   f"Soft mask applied — DB techniques upweighted.")
        else:
            agreement = "db_only"
            msg = (f"Low/medium confidence (margin={margin:.3f}). "
                   f"Showing CVE2CAPEC techniques as primary. "
                   f"Semantic result shown for reference.")

    return {
        "cve_id"         : cve_id,
        "confidence"     : confidence,
        "margin"         : margin,
        "top_techniques" : top_techniques,
        "db_techniques"  : sorted(db_tech_set),
        "agreement"      : agreement,
        "masked"         : masked,
        "message"        : msg,
    }

def print_lookup(result):
    """Pretty-print a single CVE lookup result."""
    print(f"\n{'═'*60}")
    print(f"CVE      : {result['cve_id']}")
    print(f"Confidence: {result['confidence'].upper():<8} "
          f"(margin={result['margin']:.4f})")
    print(f"Agreement : {result['agreement'].upper()}")
    print(f"BRON Mask : {'applied' if result['masked'] else 'not applied'}")
    print(f"Message   : {result['message']}")
    print(f"\nTop-{len(result['top_techniques'])} Techniques (after mask):")
    for rank, (tc, sc, name) in enumerate(result['top_techniques'], 1):
        db_flag = " ◀ in DB" if tc in result['db_techniques'] else ""
        print(f"  #{rank}  {tc:<10} score={sc:.4f}  {name[:40]}{db_flag}")
    if result['db_techniques']:
        print(f"\nCVE2CAPEC DB techniques: {result['db_techniques']}")
    print(f"{'═'*60}")

# ── 5. Smoke test on a few KEV CVEs ───────────────────────────────
print("\n── Smoke test ───────────────────────────────────────────")
sim_kev_enr_np = torch.matmul(kev_embs_enriched, tech_embeddings_enriched.T)

# pick 3 CVEs: one likely in DB, one likely not
for i in [0, 50, 100]:
    if i >= len(kev_cve_ids):
        continue
    result = lookup_cve(
        cve_id          = kev_cve_ids[i],
        cve_description = kev_texts_enriched[i],
        sim_scores_tensor = sim_kev_enr_np[i],
    )
    print_lookup(result)

print("\n✓ Lookup system ready. Ready for Cell E (batch eval).")

Fetching file list from GitHub...
Found 28 JSONL files: ['CVE-1999.jsonl', 'CVE-2000.jsonl', 'CVE-2001.jsonl', 'CVE-2002.jsonl', 'CVE-2003.jsonl']...
  ✓ CVE-1999.jsonl
  ✓ CVE-2000.jsonl
  ✓ CVE-2001.jsonl
  ✓ CVE-2002.jsonl
  ✓ CVE-2003.jsonl
  ✓ CVE-2004.jsonl
  ✓ CVE-2005.jsonl
  ✓ CVE-2006.jsonl
  ✓ CVE-2007.jsonl
  ✓ CVE-2008.jsonl
  ✓ CVE-2009.jsonl
  ✓ CVE-2010.jsonl
  ✓ CVE-2011.jsonl
  ✓ CVE-2012.jsonl
  ✓ CVE-2013.jsonl
  ✓ CVE-2014.jsonl
  ✓ CVE-2015.jsonl
  ✓ CVE-2016.jsonl
  ✓ CVE-2017.jsonl
  ✓ CVE-2018.jsonl
  ✓ CVE-2019.jsonl
  ✓ CVE-2020.jsonl
  ✓ CVE-2021.jsonl
  ✓ CVE-2022.jsonl
  ✓ CVE-2023.jsonl
  ✓ CVE-2024.jsonl
  ✓ CVE-2025.jsonl
  ✓ CVE-2026.jsonl

Downloaded: 28  Already existed: 0
Total local files: 28

Loading CVE2CAPEC database...
CVEs in database         : 341,492
CVEs with ≥1 technique   : 237,224
KEV  CVEs in DB          : 419 / 419
SMET CVEs in DB          : 302 / 302

── Smoke test ───────────────────────────────────────────

═════════════════════════

In [18]:
# Cell E (fixed) — correct device + fix mask strategy

# ── Fix 1: move tech_embeddings_enriched to device ────────────────
tech_embeddings_enriched = tech_embeddings_enriched.to(device) \
    if isinstance(tech_embeddings_enriched, torch.Tensor) \
    else torch.tensor(tech_embeddings_enriched).to(device)

# Also fix sim matrices to use device-correct computation
sim_kev  = torch.matmul(kev_embs_enriched.to(device),
                         tech_embeddings_enriched.T)
sim_smet = torch.matmul(cve_embeddings.to(device),
                         tech_embeddings.to(device).T)

# ── Fix 2: revised lookup_cve — DB boosts, never gates ────────────
def lookup_cve_v2(cve_id, cve_description,
                  sim_scores_tensor,
                  technique_ids=technique_ids,
                  db=cve2capec_db,
                  cfg=CFG_LOOKUP):
    """
    Revised strategy:
    - Hard mask  → DB techniques get +boost added to score, 
                   non-DB get small penalty. Never zeroed out.
    - Soft mask  → smaller boost, smaller penalty.
    - No mask    → pure semantic (no DB entry)
    This keeps top-10 always populated from full 214-technique space.
    """
    scores   = sim_scores_tensor.cpu().float().numpy().copy()
    db_entry = db.get(cve_id, {})
    db_techs = db_entry.get("TECHNIQUES", [])
    db_set   = set(db_techs)

    # ── Margin-based confidence ───────────────────────────────────
    top2_idx   = np.argsort(-scores)[:2]
    margin     = float(scores[top2_idx[0]] - scores[top2_idx[1]])
    if   margin >= cfg["margin_high"]: confidence = "high"
    elif margin >= cfg["margin_low"]:  confidence = "medium"
    else:                              confidence = "low"

    # ── Boost strategy (additive to cosine score) ─────────────────
    masked = False
    if db_set:
        masked = True
        if confidence == "high":
            boost      = 0.15   # strong boost for DB techniques
            penalty    = 0.05   # small penalty for non-DB
        else:
            boost      = 0.08   # softer boost
            penalty    = 0.02   # minimal penalty

        boosted = scores.copy()
        for i, tc in enumerate(technique_ids):
            if tc in db_set:
                boosted[i] += boost
            else:
                boosted[i] -= penalty
        scores = boosted

    # ── Top-K from full space ─────────────────────────────────────
    ranked   = np.argsort(-scores)
    top_k_idx = ranked[:cfg["top_k"]]

    orig_scores = sim_scores_tensor.cpu().float().numpy()
    top_techniques = [
        (technique_ids[i], round(float(orig_scores[i]), 4),
         next((k for k,v in name_to_tcode.items() if v==technique_ids[i]),
              technique_ids[i]))
        for i in top_k_idx
    ]

    # ── Agreement logic ───────────────────────────────────────────
    predicted_set = {tc for tc,_,_ in top_techniques}
    if not db_set:
        agreement = "no_db"
        msg = (f"{confidence.capitalize()} confidence (margin={margin:.3f}). "
               f"No CVE2CAPEC entry — pure semantic result.")
    elif confidence == "high":
        overlap = predicted_set & db_set
        if overlap:
            agreement = "agree"
            msg = (f"High confidence (margin={margin:.3f}). "
                   f"Semantic agrees with CVE2CAPEC on: {sorted(overlap)}. "
                   f"DB boost applied.")
        else:
            agreement = "disagree"
            msg = (f"High confidence (margin={margin:.3f}) but disagrees "
                   f"with CVE2CAPEC ({sorted(db_set)}). Showing both — "
                   f"review recommended.")
    else:
        overlap = predicted_set & db_set
        agreement = "partial" if overlap else "db_only"
        msg = (f"{confidence.capitalize()} confidence (margin={margin:.3f}). "
               f"Soft DB boost applied. "
               + (f"Overlap: {sorted(overlap)}." if overlap
                  else f"No overlap — DB suggests {sorted(db_set)}."))

    return {
        "cve_id"        : cve_id,
        "confidence"    : confidence,
        "margin"        : margin,
        "top_techniques": top_techniques,
        "db_techniques" : sorted(db_set),
        "agreement"     : agreement,
        "masked"        : masked,
        "message"       : msg,
    }

# ── Re-run batch eval with v2 ─────────────────────────────────────
def batch_lookup_eval_v2(cve_ids, descriptions, ground_truths,
                          sim_matrix, cfg, label):
    h1 = h5 = h10 = 0
    tier_counts = {"high": 0, "medium": 0, "low": 0}
    tier_hits   = {"high": [0,0,0], "medium": [0,0,0], "low": [0,0,0]}
    agree_counts = {"agree":0,"partial":0,"disagree":0,"db_only":0,"no_db":0}

    for i in range(len(cve_ids)):
        true_labels = set(ground_truths[i]) if isinstance(
            ground_truths[i], list) else ground_truths[i]
        result  = lookup_cve_v2(cve_ids[i], descriptions[i],
                                sim_matrix[i], cfg=cfg)
        top_tcs = [tc for tc,_,_ in result["top_techniques"]]
        conf    = result["confidence"]
        tier_counts[conf] += 1
        agree_counts[result["agreement"]] += 1
        hit1  = int(top_tcs[0] in true_labels)
        hit5  = int(bool(true_labels.intersection(set(top_tcs[:5]))))
        hit10 = int(bool(true_labels.intersection(set(top_tcs))))
        h1+=hit1; h5+=hit5; h10+=hit10
        tier_hits[conf][0]+=hit1
        tier_hits[conf][1]+=hit5
        tier_hits[conf][2]+=hit10

    n = len(cve_ids)
    print(f"\n── {label} ───────────────────────────────────────────")
    print(f"  R@1={h1/n*100:.2f}%  R@5={h5/n*100:.2f}%  "
          f"R@10={h10/n*100:.2f}%  (n={n})")
    print(f"  Confidence: ", end="")
    for tier in ["high","medium","low"]:
        tc = tier_counts[tier]
        if tc:
            th = tier_hits[tier]
            print(f"{tier}({tc}→R@5={th[1]/tc*100:.1f}%) ", end="")
    print(f"\n  Agreement: {dict((k,v) for k,v in agree_counts.items() if v)}")
    return {"r1":h1/n*100,"r5":h5/n*100,"r10":h10/n*100}

best_cfg = MARGIN_CONFIGS[0]   # margin_high=0.05

print("="*65)
print("LOOKUP v2 (additive boost, full candidate space)")
print("="*65)
r_kev_v2  = batch_lookup_eval_v2(
    kev_cve_ids, kev_texts_enriched, kev_ground_truths,
    sim_kev,  best_cfg, f"KEV  (n={len(kev_cve_ids)})")
r_smet_v2 = batch_lookup_eval_v2(
    smet_df["CVE_ID"].tolist(), cve_texts, cve_ground_truths,
    sim_smet, best_cfg, f"SMET (n={len(cve_texts)})")

print("\n" + "="*65)
print(f"{'Config':<38} {'R@1':>7} {'R@5':>7} {'R@10':>7}")
print("─"*65)
print(f"{'[KEV]  Rerank CE|enr (best baseline)':<38} "
      f"{'13.13%':>7} {'35.80%':>7} {'54.65%':>7}")
print(f"{'[KEV]  Lookup v2 (boost+full space)':<38} "
      f"{r_kev_v2['r1']:>6.2f}% {r_kev_v2['r5']:>6.2f}% "
      f"{r_kev_v2['r10']:>6.2f}%")
print("─"*65)
print(f"{'[SMET] Hybrid RRF|orig (best baseline)':<38} "
      f"{'32.45%':>7} {'63.91%':>7} {'76.16%':>7}")
print(f"{'[SMET] Lookup v2 (boost+full space)':<38} "
      f"{r_smet_v2['r1']:>6.2f}% {r_smet_v2['r5']:>6.2f}% "
      f"{r_smet_v2['r10']:>6.2f}%")
print("─"*65)
print(f"{'[SMET] Paper baseline':<38} "
      f"{'32.45%':>7} {'67.71%':>7} {'  —':>7}")
print("="*65)

# ── Fix interactive demo ──────────────────────────────────────────
print("\n── Interactive demo ─────────────────────────────────────")
for cve_id, desc in [
    (kev_cve_ids[0],  kev_texts_enriched[0]),
    (kev_cve_ids[50], kev_texts_enriched[50]),
    (kev_cve_ids[100],kev_texts_enriched[100]),
]:
    enc = tokenizer([desc], max_length=512, padding=True,
                    truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out  = model(**enc)
        mask = enc["attention_mask"]
        tok  = out.last_hidden_state
        emb  = (tok * mask.unsqueeze(-1).float()).sum(1) / \
               mask.float().sum(-1, keepdim=True)
        emb  = F.normalize(emb, p=2, dim=1)
    # both on same device now
    sim_row = torch.matmul(emb, tech_embeddings_enriched.T).squeeze(0)
    result  = lookup_cve_v2(cve_id, desc, sim_row, cfg=best_cfg)
    print_lookup(result)

LOOKUP v2 (additive boost, full candidate space)

── KEV  (n=419) ───────────────────────────────────────────
  R@1=6.92%  R@5=29.36%  R@10=29.36%  (n=419)
  Confidence: high(129→R@5=34.9%) medium(121→R@5=29.8%) low(169→R@5=24.9%) 
  Agreement: {'agree': 97, 'partial': 137, 'disagree': 18, 'db_only': 123, 'no_db': 44}

── SMET (n=302) ───────────────────────────────────────────
  R@1=24.83%  R@5=54.97%  R@10=54.97%  (n=302)
  Confidence: high(89→R@5=55.1%) medium(92→R@5=56.5%) low(121→R@5=53.7%) 
  Agreement: {'agree': 57, 'partial': 100, 'disagree': 21, 'db_only': 82, 'no_db': 42}

Config                                     R@1     R@5    R@10
─────────────────────────────────────────────────────────────────
[KEV]  Rerank CE|enr (best baseline)    13.13%  35.80%  54.65%
[KEV]  Lookup v2 (boost+full space)      6.92%  29.36%  29.36%
─────────────────────────────────────────────────────────────────
[SMET] Hybrid RRF|orig (best baseline)  32.45%  63.91%  76.16%
[SMET] Lookup v2 (boost+fu

In [19]:
# Cell E (v3) — Hybrid + correct top_k + tuned boosts

# ── Config: now top_k=10, try multiple boost values ───────────────
BOOST_CONFIGS = [
    dict(margin_high=0.05, margin_low=0.02,
         boost_high=0.15, penalty_high=0.05,
         boost_low=0.08,  penalty_low=0.02,  top_k=10),
    dict(margin_high=0.05, margin_low=0.02,
         boost_high=0.25, penalty_high=0.03,
         boost_low=0.12,  penalty_low=0.01,  top_k=10),
    dict(margin_high=0.05, margin_low=0.02,
         boost_high=0.35, penalty_high=0.02,
         boost_low=0.18,  penalty_low=0.01,  top_k=10),
]

def lookup_cve_v3(cve_id, cve_description,
                  sim_scores_tensor,
                  bm25_model=None,
                  technique_ids=technique_ids,
                  db=cve2capec_db,
                  cfg=BOOST_CONFIGS[0]):
    """
    Pipeline:
    1. Semantic cosine scores (pre-computed, passed in)
    2. Hybrid RRF fusion with BM25 (if bm25_model provided)
    3. DB boost (additive, post-hybrid)
    4. Confidence from margin on RAW semantic scores (before any boost)
    """
    orig_scores = sim_scores_tensor.cpu().float().numpy().copy()
    db_entry    = db.get(cve_id, {})
    db_techs    = db_entry.get("TECHNIQUES", [])
    db_set      = set(db_techs)

    # ── Step 1: confidence from raw semantic margin ───────────────
    top2_idx = np.argsort(-orig_scores)[:2]
    margin   = float(orig_scores[top2_idx[0]] - orig_scores[top2_idx[1]])
    if   margin >= cfg["margin_high"]: confidence = "high"
    elif margin >= cfg["margin_low"]:  confidence = "medium"
    else:                              confidence = "low"

    # ── Step 2: hybrid RRF if BM25 available ─────────────────────
    if bm25_model is not None:
        tokenized_q  = cve_description.lower().split()
        bm25_scores  = bm25_model.get_scores(tokenized_q)
        sem_ranks    = {idx: r+1 for r, idx in
                        enumerate(np.argsort(-orig_scores))}
        lex_ranks    = {}
        for r, idx in enumerate(np.argsort(-bm25_scores)):
            lex_ranks[idx] = float('inf') \
                if bm25_scores[idx] == 0.0 else r+1
        rrf_k = 60
        scores = np.array([
            1/(rrf_k + sem_ranks[i]) + 1/(rrf_k + lex_ranks[i])
            for i in range(len(technique_ids))
        ])
    else:
        scores = orig_scores.copy()

    # ── Step 3: DB additive boost (post-hybrid) ───────────────────
    masked = False
    if db_set:
        masked = True
        boost   = cfg["boost_high"]   if confidence == "high" \
                  else cfg["boost_low"]
        penalty = cfg["penalty_high"] if confidence == "high" \
                  else cfg["penalty_low"]
        for i, tc in enumerate(technique_ids):
            if tc in db_set:
                scores[i] += boost
            else:
                scores[i] -= penalty

    # ── Step 4: top-K from full space ────────────────────────────
    top_k_idx      = np.argsort(-scores)[:cfg["top_k"]]
    top_techniques = [
        (technique_ids[i], round(float(orig_scores[i]), 4),
         next((k for k,v in name_to_tcode.items()
               if v == technique_ids[i]), technique_ids[i]))
        for i in top_k_idx
    ]

    # ── Step 5: agreement logic ───────────────────────────────────
    predicted_set = {tc for tc,_,_ in top_techniques}
    if not db_set:
        agreement = "no_db"
        msg = (f"{confidence.capitalize()} confidence "
               f"(margin={margin:.3f}). No DB entry — pure result.")
    elif confidence == "high":
        overlap = predicted_set & db_set
        agreement = "agree" if overlap else "disagree"
        msg = (f"High confidence (margin={margin:.3f}). "
               + (f"Agrees with DB on {sorted(overlap)}."
                  if overlap else
                  f"Disagrees with DB {sorted(db_set)} — review both."))
    else:
        overlap = predicted_set & db_set
        agreement = "partial" if overlap else "db_only"
        msg = (f"{confidence.capitalize()} confidence "
               f"(margin={margin:.3f}). DB boost applied. "
               + (f"Overlap: {sorted(overlap)}."
                  if overlap else
                  f"No overlap — DB suggests {sorted(db_set)}."))

    return {"cve_id": cve_id, "confidence": confidence,
            "margin": margin, "top_techniques": top_techniques,
            "db_techniques": sorted(db_set),
            "agreement": agreement, "masked": masked, "message": msg}


def batch_eval_v3(cve_ids, descs, ground_truths,
                  sim_matrix, cfg, label, use_hybrid=False,
                  bm25_idx=None):
    h1=h5=h10=0
    tier_c = {"high":0,"medium":0,"low":0}
    tier_h = {"high":[0,0,0],"medium":[0,0,0],"low":[0,0,0]}
    ag_c   = {"agree":0,"partial":0,"disagree":0,
               "db_only":0,"no_db":0}

    for i in range(len(cve_ids)):
        true_labels = set(ground_truths[i]) if isinstance(
            ground_truths[i], list) else ground_truths[i]
        result  = lookup_cve_v3(
            cve_ids[i], descs[i], sim_matrix[i],
            bm25_model = bm25_idx if use_hybrid else None,
            cfg        = cfg)
        top_tcs = [tc for tc,_,_ in result["top_techniques"]]
        conf    = result["confidence"]
        tier_c[conf] += 1
        ag_c[result["agreement"]] += 1
        hit1  = int(top_tcs[0] in true_labels)
        hit5  = int(bool(true_labels.intersection(set(top_tcs[:5]))))
        hit10 = int(bool(true_labels.intersection(set(top_tcs[:10]))))
        h1+=hit1; h5+=hit5; h10+=hit10
        tier_h[conf][0]+=hit1
        tier_h[conf][1]+=hit5
        tier_h[conf][2]+=hit10

    n = len(cve_ids)
    mode = "Hybrid+DB" if use_hybrid else "Semantic+DB"
    print(f"\n── {label} [{mode}] ──────────────────────────────────")
    print(f"  R@1={h1/n*100:.2f}%  R@5={h5/n*100:.2f}%  "
          f"R@10={h10/n*100:.2f}%")
    print(f"  Confidence tiers:")
    for tier in ["high","medium","low"]:
        tc=tier_c[tier]
        if tc:
            th=tier_h[tier]
            print(f"    {tier:<8} n={tc:>3}  "
                  f"R@1={th[0]/tc*100:.1f}%  "
                  f"R@5={th[1]/tc*100:.1f}%  "
                  f"R@10={th[2]/tc*100:.1f}%")
    print(f"  Agreement: "
          f"{dict((k,v) for k,v in ag_c.items() if v)}")
    return {"r1":h1/n*100,"r5":h5/n*100,"r10":h10/n*100}

# ── Run all configs × both datasets × semantic + hybrid ───────────
print("="*70)
print("FULL SWEEP — Boost configs × Semantic vs Hybrid")
print("="*70)

all_rows = []
for ci, cfg in enumerate(BOOST_CONFIGS):
    tag = (f"boost_hi={cfg['boost_high']} "
           f"pen_hi={cfg['penalty_high']} "
           f"boost_lo={cfg['boost_low']}")
    print(f"\n{'━'*70}")
    print(f"Config {ci+1}: {tag}")

    for use_h, mode_tag in [(False,"Sem"), (True,"Hyb")]:
        bm25_use_kev  = bm25_enriched if use_h else None
        bm25_use_smet = bm25_model    if use_h else None
        sim_s = sim_smet if not use_h else \
                torch.matmul(cve_embeddings.to(device),
                             tech_embeddings.to(device).T)

        rk = batch_eval_v3(
            kev_cve_ids, kev_texts_enriched, kev_ground_truths,
            sim_kev, cfg, f"KEV  cfg{ci+1}",
            use_hybrid=use_h, bm25_idx=bm25_use_kev)
        rs = batch_eval_v3(
            smet_df["CVE_ID"].tolist(), cve_texts, cve_ground_truths,
            sim_s, cfg, f"SMET cfg{ci+1}",
            use_hybrid=use_h, bm25_idx=bm25_use_smet)
        all_rows.append((f"KEV  cfg{ci+1} {mode_tag}", rk))
        all_rows.append((f"SMET cfg{ci+1} {mode_tag}", rs))

# ── Final comparison table ─────────────────────────────────────────
print("\n" + "="*70)
print("FINAL TABLE")
print("="*70)
print(f"{'Config':<30} {'R@1':>8} {'R@5':>8} {'R@10':>8}")
print("─"*70)

# Baselines
for name, r1, r5, r10 in [
    ("[KEV]  Rerank CE|enr",         13.13, 35.80, 54.65),
    ("[SMET] Hybrid RRF|orig",       32.45, 63.91, 76.16),
    ("[SMET] Paper",                 32.45, 67.71,  None),
]:
    r10s = f"{r10:.2f}%" if r10 else "  —"
    print(f"{'BASELINE '+name:<30} {r1:>7.2f}% {r5:>7.2f}% {r10s:>8}")

print("─"*70)
prev_set = None
for label, r in all_rows:
    cur_set = label[:4]
    if prev_set and prev_set != cur_set:
        print("─"*70)
    print(f"{label:<30} {r['r1']:>7.2f}% {r['r5']:>7.2f}% "
          f"{r['r10']:>7.2f}%")
    prev_set = cur_set
print("="*70)

FULL SWEEP — Boost configs × Semantic vs Hybrid

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Config 1: boost_hi=0.15 pen_hi=0.05 boost_lo=0.08

── KEV  cfg1 [Semantic+DB] ──────────────────────────────────
  R@1=6.92%  R@5=29.36%  R@10=43.91%
  Confidence tiers:
    high     n=129  R@1=4.7%  R@5=34.9%  R@10=50.4%
    medium   n=121  R@1=10.7%  R@5=29.8%  R@10=39.7%
    low      n=169  R@1=5.9%  R@5=24.9%  R@10=42.0%
  Agreement: {'agree': 104, 'partial': 162, 'disagree': 11, 'db_only': 98, 'no_db': 44}

── SMET cfg1 [Semantic+DB] ──────────────────────────────────
  R@1=24.83%  R@5=54.97%  R@10=68.54%
  Confidence tiers:
    high     n= 89  R@1=33.7%  R@5=55.1%  R@10=68.5%
    medium   n= 92  R@1=20.7%  R@5=56.5%  R@10=70.7%
    low      n=121  R@1=21.5%  R@5=53.7%  R@10=66.9%
  Agreement: {'agree': 68, 'partial': 123, 'disagree': 10, 'db_only': 59, 'no_db': 42}

── KEV  cfg1 [Hybrid+DB] ──────────────────────────────────
  R@1=2.86%  R@5=10.02%  R@10=32.22%


In [20]:
# Cell F — Pre-filter strategy (Option B: restrict-then-fallback)

PREFILT_CFG = dict(
    restrict_threshold = 0.45,  # min similarity within DB pool to trust it
    fallback_threshold = 0.35,  # min similarity in full search to report
    top_k              = 10,
    margin_high        = 0.05,
    margin_low         = 0.02,
)

def lookup_cve_prefilt(cve_id, cve_description,
                       cve_embedding,              # (1, 768) tensor on device
                       technique_ids=technique_ids,
                       tech_emb_matrix=tech_embeddings_enriched,
                       db=cve2capec_db,
                       cfg=PREFILT_CFG):
    """
    Two-stage pre-filter:
    Stage 1: If DB entry → search restricted to DB techniques
             If best restricted score >= restrict_threshold → return it
    Stage 2: Full 214-technique search (fallback or no DB)
    """
    db_entry = db.get(cve_id, {})
    db_techs = db_entry.get("TECHNIQUES", [])
    db_set   = set(db_techs)

    # ── Full similarity row (we need it regardless) ───────────────
    with torch.no_grad():
        full_sim = torch.matmul(
            cve_embedding,
            tech_emb_matrix.T).squeeze(0).cpu().float().numpy()

    # ── Confidence from full similarity margin ────────────────────
    top2      = np.argsort(-full_sim)[:2]
    margin    = float(full_sim[top2[0]] - full_sim[top2[1]])
    confidence = ("high"   if margin >= cfg["margin_high"] else
                  "medium" if margin >= cfg["margin_low"]  else "low")

    # ── Stage 1: restricted search within DB techniques ───────────
    stage = "full_search"
    restricted_result = None

    if db_set:
        db_indices = [i for i, tc in enumerate(technique_ids)
                      if tc in db_set]
        if db_indices:
            db_sims  = [(technique_ids[i], full_sim[i])
                        for i in db_indices]
            db_sims  = sorted(db_sims, key=lambda x: -x[1])
            best_tc, best_score = db_sims[0]

            if best_score >= cfg["restrict_threshold"]:
                # Restricted search is confident → use it
                stage = "restricted_db"
                top_k_restricted = db_sims[:cfg["top_k"]]
                # pad with full-search results if DB has < top_k
                if len(top_k_restricted) < cfg["top_k"]:
                    seen = {tc for tc,_ in top_k_restricted}
                    for idx in np.argsort(-full_sim):
                        tc = technique_ids[idx]
                        if tc not in seen:
                            top_k_restricted.append(
                                (tc, full_sim[idx]))
                            seen.add(tc)
                        if len(top_k_restricted) >= cfg["top_k"]:
                            break
                top_techniques = [
                    (tc, round(float(sc), 4),
                     next((k for k,v in name_to_tcode.items()
                           if v == tc), tc))
                    for tc, sc in top_k_restricted[:cfg["top_k"]]
                ]
                restricted_result = best_score
            else:
                # DB exists but similarity too low → note disagreement
                stage = "db_low_sim"

    # ── Stage 2: full search fallback ─────────────────────────────
    if stage != "restricted_db":
        top_k_idx = np.argsort(-full_sim)[:cfg["top_k"]]
        top_techniques = [
            (technique_ids[i], round(float(full_sim[i]), 4),
             next((k for k,v in name_to_tcode.items()
                   if v == technique_ids[i]), technique_ids[i]))
            for i in top_k_idx
        ]

    # ── Agreement & message ───────────────────────────────────────
    predicted_set = {tc for tc,_,_ in top_techniques}
    overlap       = predicted_set & db_set

    if not db_set:
        agreement = "no_db"
        msg = (f"{confidence.capitalize()} semantic confidence "
               f"(margin={margin:.3f}). No DB entry.")
    elif stage == "restricted_db":
        agreement = "agree"
        msg = (f"DB pre-filter applied (best_sim={restricted_result:.3f} "
               f">= {cfg['restrict_threshold']}). "
               f"Showing DB-restricted results. "
               f"Semantic confidence: {confidence} (margin={margin:.3f}).")
    elif stage == "db_low_sim":
        agreement = "db_weak"
        msg = (f"DB entry exists {sorted(db_set)} but best similarity "
               f"within DB pool = {max(full_sim[i] for i,tc in enumerate(technique_ids) if tc in db_set):.3f} "
               f"< threshold {cfg['restrict_threshold']}. "
               f"Falling back to full semantic search.")
    else:
        agreement = "no_db"
        msg = f"Full semantic search (no DB entry). margin={margin:.3f}."

    return {
        "cve_id"        : cve_id,
        "confidence"    : confidence,
        "margin"        : margin,
        "stage"         : stage,
        "top_techniques": top_techniques,
        "db_techniques" : sorted(db_set),
        "agreement"     : agreement,
        "message"       : msg,
    }

# ── Batch eval ────────────────────────────────────────────────────
def batch_prefilt_eval(cve_ids, descs, ground_truths,
                       embeddings_tensor, cfg, label):
    h1=h5=h10=0
    stage_c  = {"restricted_db":0,"db_low_sim":0,"full_search":0}
    stage_h  = {"restricted_db":[0,0,0],
                "db_low_sim":[0,0,0],"full_search":[0,0,0]}

    for i in range(len(cve_ids)):
        true_labels = set(ground_truths[i]) if isinstance(
            ground_truths[i], list) else ground_truths[i]

        emb = embeddings_tensor[i:i+1]   # (1, 768)
        result = lookup_cve_prefilt(
            cve_ids[i], descs[i], emb, cfg=cfg)
        top_tcs = [tc for tc,_,_ in result["top_techniques"]]
        stg     = result["stage"]
        stage_c[stg] += 1

        hit1  = int(top_tcs[0] in true_labels)
        hit5  = int(bool(true_labels.intersection(set(top_tcs[:5]))))
        hit10 = int(bool(true_labels.intersection(set(top_tcs[:10]))))
        h1+=hit1; h5+=hit5; h10+=hit10
        stage_h[stg][0]+=hit1
        stage_h[stg][1]+=hit5
        stage_h[stg][2]+=hit10

    n = len(cve_ids)
    print(f"\n── {label} ───────────────────────────────────────────")
    print(f"  R@1={h1/n*100:.2f}%  R@5={h5/n*100:.2f}%  "
          f"R@10={h10/n*100:.2f}%  (n={n})")
    print(f"  Stage breakdown:")
    for stg in ["restricted_db","db_low_sim","full_search"]:
        sc = stage_c[stg]
        if sc:
            sh = stage_h[stg]
            print(f"    {stg:<16} n={sc:>3}  "
                  f"R@1={sh[0]/sc*100:.1f}%  "
                  f"R@5={sh[1]/sc*100:.1f}%  "
                  f"R@10={sh[2]/sc*100:.1f}%")
    return {"r1":h1/n*100,"r5":h5/n*100,"r10":h10/n*100,
            "stage_counts":stage_c}

# ── Threshold sweep ───────────────────────────────────────────────
print("="*65)
print("PRE-FILTER THRESHOLD SWEEP")
print("="*65)

sweep_results = []
for thresh in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    cfg = {**PREFILT_CFG, "restrict_threshold": thresh}
    print(f"\nrestrict_threshold={thresh}")
    rk = batch_prefilt_eval(
        kev_cve_ids, kev_texts_enriched, kev_ground_truths,
        kev_embs_enriched.to(device), cfg, f"KEV  t={thresh}")
    rs = batch_prefilt_eval(
        smet_df["CVE_ID"].tolist(), cve_texts, cve_ground_truths,
        cve_embeddings.to(device), cfg, f"SMET t={thresh}")
    sweep_results.append((thresh, rk, rs))

# ── Summary table ─────────────────────────────────────────────────
print("\n" + "="*75)
print(f"{'thresh':<10} {'KEV R@1':>8} {'KEV R@5':>8} {'KEV R@10':>9} "
      f"{'SMET R@1':>9} {'SMET R@5':>9} {'SMET R@10':>10}")
print("─"*75)
print(f"{'BASELINE':<10} {'13.13%':>8} {'35.80%':>8} {'54.65%':>9} "
      f"{'32.45%':>9} {'63.91%':>9} {'76.16%':>10}")
print("─"*75)
for thresh, rk, rs in sweep_results:
    print(f"{thresh:<10.2f} "
          f"{rk['r1']:>7.2f}% {rk['r5']:>7.2f}% {rk['r10']:>8.2f}%  "
          f"{rs['r1']:>8.2f}% {rs['r5']:>8.2f}% {rs['r10']:>9.2f}%")
print("="*75)

PRE-FILTER THRESHOLD SWEEP

restrict_threshold=0.35

── KEV  t=0.35 ───────────────────────────────────────────
  R@1=5.97%  R@5=20.76%  R@10=42.00%  (n=419)
  Stage breakdown:
    restricted_db    n=172  R@1=2.9%  R@5=10.5%  R@10=34.9%
    db_low_sim       n=203  R@1=7.9%  R@5=28.1%  R@10=48.8%
    full_search      n= 44  R@1=9.1%  R@5=27.3%  R@10=38.6%

── SMET t=0.35 ───────────────────────────────────────────
  R@1=22.19%  R@5=42.05%  R@10=59.27%  (n=302)
  Stage breakdown:
    restricted_db    n=109  R@1=23.9%  R@5=31.2%  R@10=53.2%
    db_low_sim       n=151  R@1=19.9%  R@5=46.4%  R@10=60.3%
    full_search      n= 42  R@1=26.2%  R@5=54.8%  R@10=71.4%

restrict_threshold=0.4

── KEV  t=0.4 ───────────────────────────────────────────
  R@1=7.16%  R@5=23.87%  R@10=42.48%  (n=419)
  Stage breakdown:
    restricted_db    n=118  R@1=3.4%  R@5=12.7%  R@10=34.7%
    db_low_sim       n=257  R@1=8.6%  R@5=28.4%  R@10=46.7%
    full_search      n= 44  R@1=9.1%  R@5=27.3%  R@10=38.6%

── SM

In [31]:
from sentence_transformers import SentenceTransformer, util
import json
import torch

# ==========================================
# 1. Setup Model (The Correct Way)
# ==========================================
print("Loading AttackBERT via SentenceTransformers...")
# This wrapper automatically handles tokenization and mean-pooling, giving us the .encode() method
model = SentenceTransformer("basel/Attack-BERT")

# Ensure we have our techniques to compare against (assuming parent_attack_dict is still in memory from earlier)
# If you restarted your kernel, you'll need to re-run the STIX parsing cell first!
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

print("Embedding ATT&CK technique dictionary...")
# Compute the database of vectors once
tech_embeddings = model.encode(technique_texts, convert_to_tensor=True)

# ==========================================
# 2. Load the Symbolic Mappings (ATT&CK -> NIST)
# ==========================================
def load_nist_mappings(filepath="OSRs/NIST800-53-layer-navigator/nist_800_53-rev5_attack-16.1-enterprise_json.json"):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Warning: {filepath} not found. Using mock CTID mapping for demonstration.")
        return {
            "T1190": ["AC-3 (Access Enforcement)", "SI-4 (Information System Monitoring)", "SI-3 (Malicious Code Protection)"],
            "T1059": ["CM-6 (Configuration Settings)", "AU-2 (Event Logging)"],
            "T1566": ["AT-2 (Security Awareness Training)", "SC-15 (Collaborative Computing Devices)"],
            "T1078": ["AC-2 (Account Management)", "IA-2 (Identification and Authentication)"],
            "T1195": ["SA-12 (Supply Chain Protection)", "SI-4 (Information System Monitoring)"]
        }

nist_map = load_nist_mappings()

# ==========================================
# 3. Define the End-to-End Prediction Function
# ==========================================
def predict_threat_model(cve_text, top_k_techniques=3):
    """
    Takes a raw CVE string, predicts ATT&CK techniques via Bi-Encoder,
    and maps them to NIST controls via Symbolic lookup.
    """
    # 1. Neural Retrieval (CVE -> ATT&CK)
    cve_embedding = model.encode(cve_text, convert_to_tensor=True)
    
    # Compute Cosine Similarity using SentenceTransformers utility
    cos_scores = util.cos_sim(cve_embedding, tech_embeddings)[0]
    
    # Get Top-K
    top_results = torch.topk(cos_scores, k=top_k_techniques)
    
    predicted_techniques = []
    for score, idx in zip(top_results.values, top_results.indices):
        t_code = technique_ids[idx.item()]
        predicted_techniques.append({
            "t_code": t_code,
            "score": score.item()
        })
        
    # 2. Symbolic Lookup (ATT&CK -> NIST)
    final_report = {
        "cve_text": cve_text,
        "predicted_attack_path": [],
        "recommended_nist_controls": set()
    }
    
    for tech in predicted_techniques:
        t_code = tech["t_code"]
        controls = nist_map.get(t_code, ["No direct NIST mapping found in CTID data."])
        
        final_report["predicted_attack_path"].append({
            "Technique": t_code,
            "Confidence": f"{tech['score']:.2f}"
        })
        
        for control in controls:
            final_report["recommended_nist_controls"].add(control)
            
    final_report["recommended_nist_controls"] = list(final_report["recommended_nist_controls"])
    return final_report

# ==========================================
# 4. Execute the Pipeline
# ==========================================
# A classic Supply Chain / Phishing hybrid CVE text
sample_cve = "An issue was discovered in the web interface of the router. An unauthenticated attacker can send a crafted HTTP request containing a malicious payload via a phishing email to execute arbitrary operating system commands as the root user."

print("\n" + "="*50)
print("AUTOMATED THREAT MODELING PIPELINE: EXECUTION")
print("="*50)

report = predict_threat_model(sample_cve, top_k_techniques=2)

print(f"\n[INPUT CVE]:\n{report['cve_text']}\n")

print("[NEURAL PREDICTION: ATT&CK TECHNIQUES]:")
for tech in report['predicted_attack_path']:
    print(f" -> {tech['Technique']} (Confidence Score: {tech['Confidence']})")

print("\n[SYMBOLIC LOOKUP: NIST 800-53 CONTROLS]:")
for control in report['recommended_nist_controls']:
    print(f" -> {control}")
print("="*50)

Loading AttackBERT via SentenceTransformers...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37132.60it/s]


Embedding ATT&CK technique dictionary...

AUTOMATED THREAT MODELING PIPELINE: EXECUTION

[INPUT CVE]:
An issue was discovered in the web interface of the router. An unauthenticated attacker can send a crafted HTTP request containing a malicious payload via a phishing email to execute arbitrary operating system commands as the root user.

[NEURAL PREDICTION: ATT&CK TECHNIQUES]:
 -> T1557 (Confidence Score: 0.51)
 -> T1599 (Confidence Score: 0.39)

[SYMBOLIC LOOKUP: NIST 800-53 CONTROLS]:
 -> No direct NIST mapping found in CTID data.


In [ ]:
!pip install datasets

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ------------------------------------- 527.0/527.0 kB 11.0 MB/s eta 0:00:00
     --------------------------------------- 27.5/27.5 MB 11.7 MB/s eta 0:00:00
     ---------------------------------------- 120.0/120.0 kB ? eta 0:00:00
     -------------------------------------- 144.5/144.5 kB 8.9 MB/s eta 0:00:00
     ------------------------------------- 462.9/462.9 kB 30.2 MB/s eta 0:00:00
     ---------------------------------------- 67.5/67.5 kB ? eta 0:00:00
     ---------------------------------------- 44.1/44.1 kB ? eta 0:00:00
     ---------------------------------------- 46.0/46.0 kB ? eta 0:00:00
     ---------------------------------------- 41.6/41.6 kB ? eta 0:00:00
     ---------------------------------------- 87.5/87.5 kB ? eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
